In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/YoavAndItayShared/speech/" # add path to the final project folder

Mounted at /content/drive


In [ ]:
!pip install librosa rir-generator soundfile
!pip -q install hdf5storage soundfile librosa scipy tqdm
!pip -q install hdf5storage
!pip -q install pystoi pesq torch


import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import librosa
import rir_generator as rir
import random
import glob
from torch.utils.data import Dataset, DataLoader

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Utils

In [ ]:
import os
import json
import torch
import logging
import torch.nn as nn
from torch.autograd import Variable
import numpy as np

EPSILON = np.finfo(np.float32).eps

def logger_print(log):
    logging.info(log)
    print(log)

def numParams(net):
    num = 0
    for param in net.parameters():
        if param.requires_grad:
            num += int(np.prod(param.size()))
    return num

class ToTensor(object):
    def __call__(self,
                 x,
                 type="float"):
        if type == "float":
            return torch.FloatTensor(x)
        elif type == "int":
            return torch.IntTensor(x)

def pad_to_longest(batch_data):
    """
    pad the waves with the longest length among one batch chunk
    :param batch_data:
    :return:
    """
    mix_wav_batch_list, bf_wav_batch_list, target_wav_batch_list, wav_len_list = batch_data[0]
    to_tensor = ToTensor()
    mix_wav_batch_list, bf_wav_batch_list, target_wav_batch_list = to_tensor(mix_wav_batch_list), \
                                                                   to_tensor(bf_wav_batch_list), \
                                                                   to_tensor(target_wav_batch_list)
    mix_tensor, bf_tensor, target_tensor = nn.utils.rnn.pad_sequence(mix_wav_batch_list, batch_first=True), \
                                           nn.utils.rnn.pad_sequence(bf_wav_batch_list, batch_first=True), \
                                           nn.utils.rnn.pad_sequence(target_wav_batch_list, batch_first=True)  # (B,L,M)
    return mix_tensor, bf_tensor, target_tensor, wav_len_list


class BatchInfo(object):
    def __init__(self, feats, bfs, labels, frame_mask_list):
        self.feats = feats
        self.bfs = bfs
        self.labels = labels
        self.frame_mask_list = frame_mask_list


def json_extraction(file_path, json_path, data_type):
    if not os.path.exists(json_path):
        os.makedirs(json_path)
    file_list = os.listdir(file_path)
    file_num = len(file_list)
    json_list = []

    for i in range(file_num):
        file_name = file_list[i]
        file_name = os.path.splitext(file_name)[0]
        json_list.append(file_name)

    with open(os.path.join(json_path, "{}_files.json".format(data_type)), "w") as f:
        json.dump(json_list, f, indent=4)
    return os.path.join(json_path, "{}_files.json".format(data_type))


def complex_mul(inpt1, inpt2):
    """
    inpt1: (B,2,...) or (...,2)
    inpt2: (B,2,...) or (...,2)
    """
    if inpt1.shape[1] == 2:
        out_r = inpt1[:,0,...]*inpt2[:,0,...] - inpt1[:,-1,...]*inpt2[:,-1,...]
        out_i = inpt1[:,0,...]*inpt2[:,-1,...] + inpt1[:,-1,...]*inpt2[:,0,...]
        return torch.stack((out_r, out_i), dim=1)
    elif inpt1.shape[-1] == 2:
        out_r = inpt1[...,0]*inpt2[...,0] - inpt1[...,-1]*inpt2[...,-1]
        out_i = inpt1[...,0]*inpt2[...,-1] + inpt1[...,-1]*inpt2[...,0]
        return torch.stack((out_r, out_i), dim=-1)
    else:
        raise RuntimeError("Only supports two tensor formats")

def complex_conj(inpt):
    """
    inpt: (B,2,...) or (...,2)
    """
    if inpt.shape[1] == 2:
        inpt_r, inpt_i = inpt[:,0,...], inpt[:,-1,...]
        return torch.stack((inpt_r, -inpt_i), dim=1)
    elif inpt.shape[-1] == 2:
        inpt_r, inpt_i = inpt[...,0], inpt[...,-1]
        return torch.stack((inpt_r, -inpt_i), dim=-1)

def complex_div(inpt1, inpt2):
    """
    inpt1: (B,2,...) or (...,2)
    inpt2: (B,2,...) or (...,2)
    """
    if inpt1.shape[1] == 2:
        inpt1_r, inpt1_i = inpt1[:,0,...], inpt1[:,-1,...]
        inpt2_r, inpt2_i = inpt2[:,0,...], inpt2[:,-1,...]
        denom = torch.norm(inpt2, dim=1)**2.0 + EPSILON
        out_r = inpt1_r * inpt2_r + inpt1_i * inpt2_i
        out_i = inpt1_i * inpt2_r - inpt1_r * inpt2_i
        return torch.stack((out_r/denom, out_i/denom), dim=1)
    elif inpt1.shape[-1] == 2:
        inpt1_r, inpt1_i = inpt1[...,0], inpt1[...,-1]
        inpt2_r, inpt2_i = inpt2[...,0], inpt2[...,-1]
        denom = torch.norm(inpt2, dim=-1)**2.0 + EPSILON
        out_r = inpt1_r * inpt2_r + inpt1_i * inpt2_i
        out_i = inpt1_i * inpt2_r - inpt1_r * inpt2_i
        return torch.stack((out_r/denom, out_i/denom), dim=-1)


class NormSwitch(nn.Module):
    def __init__(self,
                 norm_type: str,
                 format: str,
                 num_features: int,
                 affine: bool = True,
                 ):
        super(NormSwitch, self).__init__()
        self.norm_type = norm_type
        self.format = format
        self.num_features = num_features
        self.affine = affine

        if norm_type == "BN":
            if format == "1D":
                self.norm = nn.BatchNorm1d(num_features, affine=True)
            else:
                self.norm = nn.BatchNorm2d(num_features, affine=True)
        elif norm_type == "cLN":
            if format == "1D":
                self.norm = CumulativeLayerNorm1d(num_features, affine)
            else:
                self.norm = CumulativeLayerNorm2d(num_features, affine)
        elif norm_type == "cIN":
            if format == "2D":
                self.norm = CumulativeLayerNorm2d(num_features, affine)

    def forward(self, inpt):
        return self.norm(inpt)

class CumulativeLayerNorm2d(nn.Module):
    def __init__(self,
                 num_features,
                 affine=True,
                 eps=1e-5,
                 ):
        super(CumulativeLayerNorm2d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine

        if affine:
            self.gain = nn.Parameter(torch.ones(1,num_features,1,1))
            self.bias = nn.Parameter(torch.zeros(1,num_features,1,1))
        else:
            self.gain = Variable(torch.ones(1,num_features,1,1), requires_grad=False)
            self.bias = Variable(torch.zeros(1,num_features,1,1), requires_grad=False)

    def forward(self, inpt):
        """
        :param inpt: (B,C,T,F)
        :return:
        """
        b_size, channel, seq_len, freq_num = inpt.shape
        step_sum = inpt.sum([1,3], keepdim=True)  # (B,1,T,1)
        step_pow_sum = inpt.pow(2).sum([1,3], keepdim=True)  # (B,1,T,1)
        cum_sum = torch.cumsum(step_sum, dim=-2)  # (B,1,T,1)
        cum_pow_sum = torch.cumsum(step_pow_sum, dim=-2)  # (B,1,T,1)

        entry_cnt = np.arange(channel*freq_num, channel*freq_num*(seq_len+1), channel*freq_num)
        entry_cnt = torch.from_numpy(entry_cnt).type(inpt.type())
        entry_cnt = entry_cnt.view(1,1,seq_len,1).expand_as(cum_sum)

        cum_mean = cum_sum / entry_cnt
        cum_var = (cum_pow_sum - 2*cum_mean*cum_sum) / entry_cnt + cum_mean.pow(2)
        cum_std = (cum_var + self.eps).sqrt()

        x = (inpt - cum_mean) / cum_std
        return x * self.gain.expand_as(x).type(x.type()) + self.bias.expand_as(x).type(x.type())


class CumulativeInstanceNorm2d(nn.Module):
    def __init__(self,
                 num_features,
                 affine=True,
                 eps=1e-5,
                 ):
        super(CumulativeInstanceNorm2d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine

        if affine:
            self.gain = nn.Parameter(torch.ones(1,num_features,1,1))
            self.bias = nn.Parameter(torch.zeros(1,num_features,1,1))
        else:
            self.gain = Variable(torch.ones(1,num_features,1,1), requires_grad=False)
            self.bias = Variable(torch.zeros(1,num_features,1,1), requires_grad=False)

    def forward(self, inpt):
        """
        :param inpt: (B,C,T,F)
        :return:
        """
        b_size, channel, seq_len, freq_num = inpt.shape
        step_sum = inpt.sum([3], keepdim=True)  # (B,C,T,1)
        step_pow_sum = inpt.pow(2).sum([3], keepdim=True)  # (B,C,T,1)
        cum_sum = torch.cumsum(step_sum, dim=-2)  # (B,C,T,1)
        cum_pow_sum = torch.cumsum(step_pow_sum, dim=-2)  # (B,C,T,1)

        entry_cnt = np.arange(freq_num, freq_num*(seq_len+1), freq_num)
        entry_cnt = torch.from_numpy(entry_cnt).type(inpt.type())
        entry_cnt = entry_cnt.view(1,1,seq_len,1).expand_as(cum_sum)

        cum_mean = cum_sum / entry_cnt
        cum_var = (cum_pow_sum - 2*cum_mean*cum_sum) / entry_cnt + cum_mean.pow(2)
        cum_std = (cum_var + self.eps).sqrt()

        x = (inpt - cum_mean) / cum_std
        return x * self.gain.expand_as(x).type(x.type()) + self.bias.expand_as(x).type(x.type())


class CumulativeLayerNorm1d(nn.Module):
    def __init__(self,
                 num_features,
                 affine=True,
                 eps=1e-5,
                 ):
        super(CumulativeLayerNorm1d, self).__init__()
        self.num_features = num_features
        self.affine = affine
        self.eps = eps

        if affine:
            self.gain = nn.Parameter(torch.ones(1,num_features,1), requires_grad=True)
            self.bias = nn.Parameter(torch.zeros(1,num_features,1), requires_grad=True)
        else:
            self.gain = Variable(torch.ones(1, num_features, 1), requires_grad=False)
            self.bias = Variable(torch.zeros(1, num_features, 1), requires_gra=False)

    def forward(self, inpt):
        # inpt: (B,C,T)
        b_size, channel, seq_len = inpt.shape
        cum_sum = torch.cumsum(inpt.sum(1), dim=1)  # (B,T)
        cum_power_sum = torch.cumsum(inpt.pow(2).sum(1), dim=1)  # (B,T)

        entry_cnt = np.arange(channel, channel*(seq_len+1), channel)
        entry_cnt = torch.from_numpy(entry_cnt).type(inpt.type())
        entry_cnt = entry_cnt.view(1, -1).expand_as(cum_sum)  # (B,T)

        cum_mean = cum_sum / entry_cnt  # (B,T)
        cum_var = (cum_power_sum - 2*cum_mean*cum_sum) / entry_cnt + cum_mean.pow(2)
        cum_std = (cum_var + self.eps).sqrt()

        x = (inpt - cum_mean.unsqueeze(dim=1).expand_as(inpt)) / cum_std.unsqueeze(dim=1).expand_as(inpt)
        return x * self.gain.expand_as(x).type(x.type()) + self.bias.expand_as(x).type(x.type())

def sisnr(est, label):
    label_power = np.sum(label**2.0) + 1e-8
    scale = np.sum(est*label) / label_power
    est_true = scale * label
    est_res = est - est_true
    true_power = np.sum(est_true**2.0, axis=0) + 1e-8
    res_power = np.sum(est_res**2.0, axis=0) + 1e-8
    sdr = 10*np.log10(true_power) - 10*np.log10(res_power)
    return sdr

def cal_pesq(id, esti_utts, clean_utts, fs):
    clean_utt, esti_utt = clean_utts[id,:], esti_utts[id,:]
    from pesq import pesq
    pesq_score = pesq(fs, clean_utt, esti_utt, "nb")
    return pesq_score

def cal_stoi(id, esti_utts, clean_utts, fs):
    clean_utt, esti_utt = clean_utts[id,:], esti_utts[id,:]
    from pystoi import stoi
    stoi_score = stoi(clean_utt, esti_utt, fs, extended=True)
    return 100*stoi_score

def cal_sisnr(id, esti_utts, clean_utts, fs):
    clean_utt, esti_utt = clean_utts[id,:], esti_utts[id,:]
    sisnr_score = sisnr(esti_utt, clean_utt)
    return sisnr_score

import torch

class SpatialFilterLoss(object):
    def __init__(self, alpha, l_type):
        self.alpha = alpha
        self.l_type = l_type

    def __call__(self, resi, frame_list):
        """
        resi: (B,T,F,2), frame_list: list
        """
        b_size, seq_len, freq_num, _ = resi.shape
        mask_for_loss = []
        with torch.no_grad():
            for i in range(b_size):
                tmp_mask = torch.ones((frame_list[i], freq_num, 2), dtype=resi.dtype)
                mask_for_loss.append(tmp_mask)
            mask_for_loss = torch.nn.utils.rnn.pad_sequence(mask_for_loss, batch_first=True).to(resi.device)
            mag_mask_for_loss = mask_for_loss[...,0]

        resi_mag = torch.norm(resi, dim=-1)
        if self.l_type == "L1" or self.l_type == "l1":
            loss_com = (torch.abs(resi) * mask_for_loss).sum() / mask_for_loss.sum()
            loss_mag = (torch.abs(resi_mag) * mag_mask_for_loss).sum() / mag_mask_for_loss.sum()
        elif self.l_type == "L2" or self.l_type == "l2":
            loss_com = (torch.square(resi) * mask_for_loss).sum() / mask_for_loss.sum()
            loss_mag = (torch.square(resi_mag) * mag_mask_for_loss).sum() / mag_mask_for_loss.sum()
        else:
            raise RuntimeError("only L1 and L2 are supported")
        return self.alpha * loss_com + (1 - self.alpha) * loss_mag


class ComMagEuclideanLoss(object):
    def __init__(self, alpha, l_type):
        self.alpha = alpha
        self.l_type = l_type

    def __call__(self, est, label, frame_list):
        """
            est: (B,T,F,2)
            label: (B,T,F,2)
            frame_list: list
            alpha: scalar
            l_type: str, L1 or L2
            """
        b_size, seq_len, freq_num, _ = est.shape
        mask_for_loss = []
        with torch.no_grad():
            for i in range(b_size):
                tmp_mask = torch.ones((frame_list[i], freq_num, 2), dtype=est.dtype)
                mask_for_loss.append(tmp_mask)
            mask_for_loss = torch.nn.utils.rnn.pad_sequence(mask_for_loss, batch_first=True).to(est.device)
            mag_mask_for_loss = mask_for_loss[...,0]
        est_mag, label_mag = torch.norm(est, dim=-1), torch.norm(label, dim=-1)

        if self.l_type == "L1" or self.l_type == "l1":
            loss_com = (torch.abs(est - label) * mask_for_loss).sum() / mask_for_loss.sum()
            loss_mag = (torch.abs(est_mag - label_mag) * mag_mask_for_loss).sum() / mag_mask_for_loss.sum()
        elif self.l_type == "L2" or self.l_type == "l2":
            loss_com = (torch.square(est - label) * mask_for_loss).sum() / mask_for_loss.sum()
            loss_mag = (torch.square(est_mag - label_mag) * mag_mask_for_loss).sum() / mag_mask_for_loss.sum()
        else:
            raise RuntimeError("only L1 and L2 are supported!")
        return self.alpha * loss_com + (1 - self.alpha) * loss_mag

## Tensor_Complex

In [ ]:
import numbers
from typing import Union, List

import numpy
import torch
EPSILON = torch.finfo(torch.float32).eps

__all__ = ["ComplexTensor"]


class ComplexTensor:
    def __init__(
        self, real: Union[torch.Tensor, numpy.ndarray], imag=None, device=None
    ):
        if imag is None:
            if isinstance(real, numpy.ndarray):
                if real.dtype.kind == "c":
                    imag = real.imag
                    real = real.real
                else:
                    imag = numpy.zeros_like(real)
            elif isinstance(real, ComplexTensor):
                imag = real.imag
                real = real.real
            else:
                imag = torch.zeros_like(real, device=device)

        if isinstance(real, numpy.ndarray):
            real = torch.as_tensor(real, device=device)
        else:
            real = real.to(device)
        if isinstance(imag, numpy.ndarray):
            imag = torch.as_tensor(imag, device=device)
        else:
            imag = imag.to(device)

        if not torch.is_tensor(real):
            raise TypeError(
                f"The first arg must be torch.Tensor" f"but got {type(real)}"
            )

        if not torch.is_tensor(imag):
            raise TypeError(
                f"The second arg must be torch.Tensor" f"but got {type(imag)}"
            )
        if not real.size() == imag.size():
            raise ValueError(
                f"The two inputs must have same sizes: "
                f"{real.size()} != {imag.size()}"
            )

        self.real = real
        self.imag = imag

    def __getitem__(self, item) -> "ComplexTensor":
        return ComplexTensor(self.real[item], self.imag[item])

    def __setitem__(
        self, item, value: Union["ComplexTensor", torch.Tensor, numbers.Number]
    ):
        if isinstance(value, (ComplexTensor, complex)):
            self.real[item] = value.real
            self.imag[item] = value.imag
        else:
            self.real[item] = value
            self.imag[item] = 0

    def __mul__(
        self, other: Union["ComplexTensor", torch.Tensor, numbers.Number]
    ) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            return ComplexTensor(
                self.real * other.real - self.imag * other.imag,
                self.real * other.imag + self.imag * other.real,
            )
        else:
            return ComplexTensor(self.real * other, self.imag * other)

    def __rmul__(
        self, other: Union["ComplexTensor", torch.Tensor, numbers.Number]
    ) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            return ComplexTensor(
                other.real * self.real - other.imag * self.imag,
                other.imag * self.real + other.real * self.imag,
            )
        else:
            return ComplexTensor(other * self.real, other * self.imag)

    def __imul__(self, other):
        if isinstance(other, (ComplexTensor, numbers.Complex)):
            t = self * other
            self.real = t.real
            self.imag = t.imag
        else:
            self.real *= other
            self.imag *= other
        return self

    def __truediv__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            den = other.real ** 2 + other.imag ** 2 + EPSILON
            return ComplexTensor(
                (self.real * other.real + self.imag * other.imag) / den,
                (-self.real * other.imag + self.imag * other.real) / den,
            )
        else:
            return ComplexTensor(self.real / other, self.imag / other)

    def __rtruediv__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            den = self.real ** 2 + self.imag ** 2
            return ComplexTensor(
                (other.real * self.real + other.imag * self.imag) / den,
                (-other.real * self.imag + other.imag * self.real) / den,
            )
        else:
            den = self.real ** 2 + self.imag ** 2
            return ComplexTensor(other * self.real / den, -other * self.imag / den)

    def __itruediv__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, numbers.Complex)):
            t = self / other
            self.real = t.real
            self.imag = t.imag
        else:
            self.real /= other
            self.imag /= other
        return self

    def __add__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            return ComplexTensor(self.real + other.real, self.imag + other.imag)
        else:
            return ComplexTensor(self.real + other, self.imag)

    def __radd__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            return ComplexTensor(other.real + self.real, other.imag + self.imag)
        else:
            return ComplexTensor(other + self.real, self.imag)

    def __iadd__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            self.real += other.real
            self.imag += other.imag
        else:
            self.real += other
        return self

    def __sub__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            return ComplexTensor(self.real - other.real, self.imag - other.imag)
        else:
            return ComplexTensor(self.real - other, self.imag)

    def __rsub__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            return ComplexTensor(other.real - self.real, other.imag - self.imag)
        else:
            return ComplexTensor(other - self.real, self.imag)

    def __isub__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, complex)):
            self.real -= other.real
            self.imag -= other.imag
        else:
            self.real -= other
        return self

    def __matmul__(self, other) -> "ComplexTensor":
        if isinstance(other, ComplexTensor):
            o_real = torch.matmul(self.real, other.real) - torch.matmul(
                self.imag, other.imag
            )
            o_imag = torch.matmul(self.real, other.imag) + torch.matmul(
                self.imag, other.real
            )
        else:
            o_real = torch.matmul(self.real, other)
            o_imag = torch.matmul(self.imag, other)
        return ComplexTensor(o_real, o_imag)

    def __rmatmul__(self, other) -> "ComplexTensor":
        if isinstance(other, ComplexTensor):
            o_real = torch.matmul(other.real, self.real) - torch.matmul(
                other.imag, self.imag
            )
            o_imag = torch.matmul(other.real, self.imag) + torch.matmul(
                other.imag, self.real
            )
        else:
            o_real = torch.matmul(other, self.real)
            o_imag = torch.matmul(other, self.imag)
        return ComplexTensor(o_real, o_imag)

    def __imatmul__(self, other) -> "ComplexTensor":
        if isinstance(other, (ComplexTensor, numbers.Complex)):
            t = self @ other
            self.real = t.real
            self.imag = t.imag
        else:
            self.real @= other
            self.imag @= other
        return self

    def __neg__(self) -> "ComplexTensor":
        return ComplexTensor(-self.real, -self.imag)

    def __eq__(self, other) -> torch.Tensor:
        if isinstance(other, (ComplexTensor, complex)):
            return (self.real == other.real) ** (self.imag == other.imag)
        else:
            return (self.real == other) ** (self.imag == 0)

    def __len__(self) -> int:
        return len(self.real)

    def __repr__(self) -> str:
        import textwrap

        return (
            "ComplexTensor("
            + "\n    real="
            + textwrap.indent(repr(self.real), " " * len("    real=")).lstrip(" ")
            + ",\n    imag="
            + textwrap.indent(repr(self.imag), " " * len("    imag=")).lstrip(" ")
            + ",\n)"
        )

    def __abs__(self) -> torch.Tensor:
        return (self.real * self.real + self.imag * self.imag).sqrt()

    def __pow__(self, exponent) -> "ComplexTensor":
        if exponent == -2:
            return 1 / (self * self)
        if exponent == -1:
            return 1 / self
        if exponent == 0:
            return ComplexTensor(torch.ones_like(self.real))
        if exponent == 1:
            return self.clone()
        if exponent == 2:
            return self * self

        _abs = self.abs().pow(exponent)
        _angle = exponent * self.angle()
        return ComplexTensor(_abs * torch.cos(_angle), _abs * torch.sin(_angle))

    def __ipow__(self, exponent) -> "ComplexTensor":
        c = self ** exponent
        self.real = c.real
        self.imag = c.imag
        return self

    def abs(self) -> torch.Tensor:
        return (self.real * self.real + self.imag * self.imag).sqrt()

    def angle(self) -> torch.Tensor:
        return torch.atan2(self.imag, self.real)

    def backward(self) -> None:
        self.real.backward()
        self.imag.backward()

    def byte(self) -> "ComplexTensor":
        return ComplexTensor(self.real.byte(), self.imag.byte())

    def clone(self) -> "ComplexTensor":
        return ComplexTensor(self.real.clone(), self.imag.clone())

    def conj(self) -> "ComplexTensor":
        return ComplexTensor(self.real, -self.imag)

    def conj_(self) -> "ComplexTensor":
        self.imag.neg_()
        return self

    def contiguous(self) -> "ComplexTensor":
        return ComplexTensor(self.real.contiguous(), self.imag.contiguous())

    def copy_(self) -> "ComplexTensor":
        self.real = self.real.copy_()
        self.imag = self.imag.copy_()
        return self

    def cpu(self) -> "ComplexTensor":
        return ComplexTensor(self.real.cpu(), self.imag.cpu())

    def cuda(self) -> "ComplexTensor":
        return ComplexTensor(self.real.cuda(), self.imag.cuda())

    def expand(self, *sizes):
        return ComplexTensor(self.real.expand(*sizes), self.imag.expand(*sizes))

    def expand_as(self, *args, **kwargs):
        return ComplexTensor(
            self.real.expand_as(*args, **kwargs), self.imag.expand_as(*args, **kwargs)
        )

    def detach(self) -> "ComplexTensor":
        return ComplexTensor(self.real.detach(), self.imag.detach())

    def detach_(self) -> "ComplexTensor":
        self.real.detach_()
        self.imag.detach_()
        return self

    @property
    def device(self):
        assert self.real.device == self.imag.device
        return self.real.device

    def diag(self) -> "ComplexTensor":
        return ComplexTensor(self.real.diag(), self.imag.diag())

    def diagonal(self) -> "ComplexTensor":
        return ComplexTensor(self.real.diag(), self.imag.diag())

    def dim(self) -> int:
        return self.real.dim()

    def double(self) -> "ComplexTensor":
        return ComplexTensor(self.real.double(), self.imag.double())

    @property
    def dtype(self) -> torch.dtype:
        # Warning: Try to never use this dtype property.
        #          It will break your code, when you change to the native
        #          complex type.
        #          Use instead directly `complex_tensor.real.dtype`.
        return self.real.dtype

    def is_floating_point(self):
        return False

    def is_complex(self):
        return True

    def eq(self, other) -> torch.Tensor:
        if isinstance(other, (ComplexTensor, complex)):
            return (self.real == other.real) * (self.imag == other.imag)
        else:
            return (self.real == other) * (self.imag == 0)

    def equal(self, other) -> bool:
        if isinstance(other, (ComplexTensor, complex)):
            return self.real.equal(other.real) and self.imag.equal(other.imag)
        else:
            return self.real.equal(other) and self.imag.equal(0)

    def float(self) -> "ComplexTensor":
        return ComplexTensor(self.real.float(), self.imag.float())

    def fill(self, value) -> "ComplexTensor":
        if isinstance(value, complex):
            return ComplexTensor(self.real.fill(value.real), self.imag.fill(value.imag))
        else:
            return ComplexTensor(self.real.fill(value), self.imag.fill(0))

    def fill_(self, value) -> "ComplexTensor":
        if isinstance(value, complex):
            self.real.fill_(value.real)
            self.imag.fill_(value.imag)
        else:
            self.real.fill_(value)
            self.imag.fill_(0)
        return self

    def gather(self, dim, index) -> "ComplexTensor":
        return ComplexTensor(self.real.gather(dim, index), self.real.gather(dim, index))

    def get_device(self, *args, **kwargs):
        return self.real.get_device(*args, **kwargs)

    def half(self) -> "ComplexTensor":
        return ComplexTensor(self.real.half(), self.imag.half())

    def index_add(self, dim, index, tensor) -> "ComplexTensor":
        return ComplexTensor(
            self.real.index_add(dim, index, tensor),
            self.imag.index_add(dim, index, tensor),
        )

    def index_copy(self, dim, index, tensor) -> "ComplexTensor":
        return ComplexTensor(
            self.real.index_copy(dim, index, tensor),
            self.imag.index_copy(dim, index, tensor),
        )

    def index_fill(self, dim, index, value) -> "ComplexTensor":
        return ComplexTensor(
            self.real.index_fill(dim, index, value),
            self.imag.index_fill(dim, index, value),
        )

    def index_select(self, dim, index) -> "ComplexTensor":
        return ComplexTensor(
            self.real.index_select(dim, index), self.imag.index_select(dim, index)
        )

    def inverse(self, ntry=5) -> "ComplexTensor":
        # m x n x n
        in_size = self.size()
        a = self.view(-1, self.size(-1), self.size(-1))
        # see "The Matrix Cookbook" (http://www2.imm.dtu.dk/pubdb/p.php?3274)
        # "Section 4.3"
        for i in range(ntry):
            t = i * 0.1

            e = a.real + t * a.imag
            f = a.imag - t * a.real

            try:
                x = torch.matmul(f, e.inverse())
                z = (e + torch.matmul(x, f)).inverse()
            except Exception:
                if i == ntry - 1:
                    raise
                continue

            if t != 0.0:
                eye = torch.eye(
                    a.real.size(-1), dtype=a.real.dtype, device=a.real.device
                )[None]
                o_real = torch.matmul(z, (eye - t * x))
                o_imag = -torch.matmul(z, (t * eye + x))
            else:
                o_real = z
                o_imag = -torch.matmul(z, x)

            o = ComplexTensor(o_real, o_imag)
            return o.view(*in_size)

    def inverse2(self) -> "ComplexTensor":
        # To avoid cyclic import
        return real_matrix2complex_matrix(complex_matrix2real_matrix(self).inverse())

    def item(self) -> numbers.Number:
        return self.real.item() + 1j * self.imag.item()

    def masked_fill(self, mask, value) -> "ComplexTensor":
        if isinstance(value, complex):
            return ComplexTensor(
                self.real.masked_fill(mask, value.real),
                self.imag.masked_fill(mask, value.imag),
            )

        else:
            return ComplexTensor(
                self.real.masked_fill(mask, value), self.imag.masked_fill(mask, 0)
            )

    def masked_fill_(self, mask, value) -> "ComplexTensor":
        if isinstance(value, complex):
            self.real.masked_fill_(mask, value.real)
            self.imag.masked_fill_(mask, value.imag)
        else:
            self.real.masked_fill_(mask, value)
            self.imag.masked_fill_(mask, 0)
        return self

    def mean(self, *args, **kwargs) -> "ComplexTensor":
        return ComplexTensor(
            self.real.mean(*args, **kwargs), self.imag.mean(*args, **kwargs)
        )

    def neg(self) -> "ComplexTensor":
        return ComplexTensor(-self.real, -self.imag)

    def neg_(self) -> "ComplexTensor":
        self.real.neg_()
        self.imag.neg_()
        return self

    def nelement(self) -> int:
        return self.real.nelement()

    def numel(self) -> int:
        return self.real.numel()

    def new(self, *args, **kwargs) -> "ComplexTensor":
        return ComplexTensor(
            self.real.new(*args, **kwargs), self.imag.new(*args, **kwargs)
        )

    def new_empty(
        self, size, dtype=None, device=None, requires_grad=False
    ) -> "ComplexTensor":
        real = self.real.new_empty(
            size, dtype=dtype, device=device, requires_grad=requires_grad
        )
        imag = self.imag.new_empty(
            size, dtype=dtype, device=device, requires_grad=requires_grad
        )
        return ComplexTensor(real, imag)

    def new_full(
        self, size, fill_value, dtype=None, device=None, requires_grad=False
    ) -> "ComplexTensor":
        if isinstance(fill_value, complex):
            real_value = fill_value.real
            imag_value = fill_value.imag
        else:
            real_value = fill_value
            imag_value = 0.0

        real = self.real.new_full(
            size,
            fill_value=real_value,
            dtype=dtype,
            device=device,
            requires_grad=requires_grad,
        )
        imag = self.imag.new_full(
            size,
            fill_value=imag_value,
            dtype=dtype,
            device=device,
            requires_grad=requires_grad,
        )
        return ComplexTensor(real, imag)

    def new_tensor(
        self, data, dtype=None, device=None, requires_grad=False
    ) -> "ComplexTensor":
        if isinstance(data, ComplexTensor):
            real = data.real
            imag = data.imag
        elif isinstance(data, numpy.ndarray):
            if data.dtype.kind == "c":
                real = data.real
                imag = data.imag
            else:
                real = data
                imag = None
        else:
            real = data
            imag = None

        real = self.real.new_tensor(
            real, dtype=dtype, device=device, requires_grad=requires_grad
        )
        if imag is None:
            imag = torch.zeros_like(
                real, dtype=dtype, device=device, requires_grad=requires_grad
            )
        else:
            imag = self.imag.new_tensor(
                imag, dtype=dtype, device=device, requires_grad=requires_grad
            )
        return ComplexTensor(real, imag)

    def numpy(self) -> numpy.ndarray:
        return self.real.numpy() + 1j * self.imag.numpy()

    def __array__(self):
        # https://numpy.org/devdocs/user/basics.dispatch.html
        return self.real.__array__() + 1j * self.imag.__array__()

    def permute(self, *dims) -> "ComplexTensor":
        return ComplexTensor(self.real.permute(*dims), self.imag.permute(*dims))

    @property
    def T(self):
        return ComplexTensor(self.real.T, self.imag.T)

    def pow(self, exponent) -> "ComplexTensor":
        return self ** exponent

    def requires_grad_(self) -> "ComplexTensor":
        self.real.requires_grad_()
        self.imag.requires_grad_()
        return self

    @property
    def requires_grad(self):
        assert self.real.requires_grad == self.imag.requires_grad
        return self.real.requires_grad

    @requires_grad.setter
    def requires_grad(self, value):
        self.real.requires_grad = value
        self.imag.requires_grad = value

    def repeat(self, *sizes):
        return ComplexTensor(self.real.repeat(*sizes), self.imag.repeat(*sizes))

    def reshape(self, *shape):
        return ComplexTensor(self.real.reshape(*shape), self.imag.reshape(*shape))

    def retain_grad(self) -> "ComplexTensor":
        self.real.retain_grad()
        self.imag.retain_grad()
        return self

    def share_memory_(self) -> "ComplexTensor":
        self.real.share_memory_()
        self.imag.share_memory_()
        return self

    @property
    def shape(self) -> torch.Size:
        return self.real.shape

    def size(self, *args, **kwargs) -> torch.Size:
        return self.real.size(*args, **kwargs)

    def ndimension(self):
        return self.real.ndimension()

    @property
    def ndim(self):
        return self.real.ndim

    def sqrt(self) -> "ComplexTensor":
        return self ** 0.5

    def squeeze(self, dim=None) -> "ComplexTensor":
        if dim is None:
            return ComplexTensor(self.real.squeeze(), self.imag.squeeze())
        else:
            return ComplexTensor(self.real.squeeze(dim), self.imag.squeeze(dim))

    def sum(self, *args, **kwargs) -> "ComplexTensor":
        """
        sum(self, dim, keepdim, *, dtype=None)
        sum(self, axis, keepdims, *, dtype=None)  # numpy style

        Args:
            dim or axis:
            keepdim or keepdims:
            **kwargs:

        Returns:

        """
        return ComplexTensor(
            self.real.sum(*args, **kwargs), self.imag.sum(*args, **kwargs)
        )

    def take(self, indices) -> "ComplexTensor":
        return ComplexTensor(self.real.take(indices), self.imag.take(indices))

    def to(self, *args, **kwargs) -> "ComplexTensor":
        return ComplexTensor(
            self.real.to(*args, **kwargs), self.imag.to(*args, **kwargs)
        )

    def tolist(self) -> List[numbers.Number]:
        return [r + 1j * i for r, i in zip(self.real.tolist(), self.imag.tolist())]

    def transpose(self, dim0, dim1) -> "ComplexTensor":
        return ComplexTensor(
            self.real.transpose(dim0, dim1), self.imag.transpose(dim0, dim1)
        )

    def transpose_(self, dim0, dim1) -> "ComplexTensor":
        self.real.transpose_(dim0, dim1)
        self.imag.transpose_(dim0, dim1)
        return self

    def type(self, *args, **kwargs) -> str:
        if len(args) == 0 and len(kwargs) == 0:
            return self.real.type()
        else:
            return ComplexTensor(
                self.real.type(*args, **kwargs), self.imag.type(*args, **kwargs)
            )

    def unbind(self, dim=0) -> "ComplexTensor":
        return tuple(
            map(
                lambda x: ComplexTensor(*x),
                zip(self.real.unbind(dim=dim), self.imag.unbind(dim=dim))
            )
        )

    def unfold(self, dim, size, step):
        return ComplexTensor(
            self.real.unfold(dim, size, step), self.imag.unfold(dim, size, step)
        )

    def unsqueeze(self, dim) -> "ComplexTensor":
        return ComplexTensor(self.real.unsqueeze(dim), self.imag.unsqueeze(dim))

    def unsqueeze_(self, dim) -> "ComplexTensor":
        self.real.unsqueeze_(dim)
        self.imag.unsqueeze_(dim)
        return self

    def view(self, *args, **kwargs) -> "ComplexTensor":
        return ComplexTensor(
            self.real.view(*args, **kwargs), self.imag.view(*args, **kwargs)
        )

    def view_as(self, tensor):
        return self.view(tensor.size())

from distutils.version import LooseVersion
import functools
from typing import Sequence
from typing import Union

import torch
from torch.nn import functional as F

__all__ = [
    "einsum",
    "cat",
    "stack",
    "pad",
    "squeeze",
    "reverse",
    "trace",
    "allclose",
    "matmul",
    "solve",
]


def _fcomplex(func, nthargs=0):
    @functools.wraps(func)
    def wrapper(*args, **kwargs) -> Union[ComplexTensor, torch.Tensor]:
        signal = args[nthargs]
        if isinstance(signal, ComplexTensor):
            real_args = args[:nthargs] + (signal.real,) + args[nthargs + 1 :]
            imag_args = args[:nthargs] + (signal.imag,) + args[nthargs + 1 :]
            real = func(*real_args, **kwargs)
            imag = func(*imag_args, **kwargs)
            return ComplexTensor(real, imag)
        else:
            return func(*args, **kwargs)

    return wrapper


def einsum(equation, *operands):
    """Einsum

    >>> import numpy
    >>> def get(*shape):
    ...     real = numpy.random.rand(*shape)
    ...     imag = numpy.random.rand(*shape)
    ...     return real + 1j * imag
    >>> x = get(3, 4, 5)
    >>> y = get(3, 5, 6)
    >>> z = get(3, 6, 7)
    >>> test = einsum('aij,ajk,akl->ail',
    ...               [ComplexTensor(x), ComplexTensor(y), ComplexTensor(z)])
    >>> valid = numpy.einsum('aij,ajk,akl->ail', x, y, z)
    >>> numpy.testing.assert_allclose(test.numpy(), valid)
    >>> _ = einsum('aij->ai', ComplexTensor(x))
    >>> _ = einsum('aij->ai', [ComplexTensor(x)])

    """
    if len(operands) == 1 and isinstance(operands[0], (tuple, list)):
        operands = operands[0]

    x = operands[0]
    if isinstance(x, ComplexTensor):
        real_operands = [[x.real]]
        imag_operands = [[x.imag]]
    else:
        real_operands = [[x]]
        imag_operands = []

    for x in operands[1:]:
        if isinstance(x, ComplexTensor):
            real_operands, imag_operands = (
                [ops + [x.real] for ops in real_operands]
                + [ops + [-x.imag] for ops in imag_operands],
                [ops + [x.imag] for ops in real_operands]
                + [ops + [x.real] for ops in imag_operands],
            )
        else:
            real_operands = [ops + [x] for ops in real_operands]
            imag_operands = [ops + [x] for ops in imag_operands]

    real = sum([torch.einsum(equation, ops) for ops in real_operands])
    imag = sum([torch.einsum(equation, ops) for ops in imag_operands])
    return ComplexTensor(real, imag)


def cat(seq: Sequence[Union[ComplexTensor, torch.Tensor]], *args, **kwargs):
    """
    cat(seq, dim=0, *, out=None)
    cat(seq, axis=0, *, out=None)
    """
    reals = [v.real if isinstance(v, ComplexTensor) else v for v in seq]
    imags = [
        v.imag if isinstance(v, ComplexTensor) else torch.zeros_like(v.real)
        for v in seq
    ]
    out = kwargs.pop("out", None)
    if out is not None:
        out = out
        out_real = out.real
        out_imag = out.imag
    else:
        out_real = out_imag = None
    return ComplexTensor(
        torch.cat(reals, *args, out=out_real, **kwargs),
        torch.cat(imags, *args, out=out_imag, **kwargs),
    )


def stack(seq: Sequence[Union[ComplexTensor, torch.Tensor]], *args, **kwargs):
    """
    stack(tensors, dim=0, * out=None)
    stack(tensors, axis=0, * out=None)

    """
    reals = [v.real if isinstance(v, ComplexTensor) else v for v in seq]
    imags = [
        v.imag if isinstance(v, ComplexTensor) else torch.zeros_like(v.real)
        for v in seq
    ]

    out = kwargs.pop("out", None)
    if out is not None:
        out_real = out.real
        out_imag = out.imag
    else:
        out_real = out_imag = None
    return ComplexTensor(
        torch.stack(reals, *args, out=out_real, **kwargs),
        torch.stack(imags, *args, out=out_imag, **kwargs),
    )


pad = _fcomplex(F.pad)
squeeze = _fcomplex(torch.squeeze)


@_fcomplex
def reverse(tensor: torch.Tensor, dim=0) -> torch.Tensor:
    # https://discuss.pytorch.org/t/how-to-reverse-a-torch-tensor/382
    idx = [i for i in range(tensor.size(dim) - 1, -1, -1)]
    idx = torch.LongTensor(idx).to(tensor.device)
    inverted_tensor = tensor.index_select(dim, idx)
    return inverted_tensor


@_fcomplex
def signal_frame(
    signal: torch.Tensor, frame_length: int, frame_step: int, pad_value=0
) -> torch.Tensor:
    """Expands signal into frames of frame_length.

    Args:
        signal : (B * F, D, T)
    Returns:
        torch.Tensor: (B * F, D, T, W)
    """
    signal = F.pad(signal, (0, frame_length - 1), "constant", pad_value)
    indices = sum(
        [
            list(range(i, i + frame_length))
            for i in range(0, signal.size(-1) - frame_length + 1, frame_step)
        ],
        [],
    )

    signal = signal[..., indices].view(*signal.size()[:-1], -1, frame_length)
    return signal


def trace(a: ComplexTensor) -> ComplexTensor:
    if LooseVersion(torch.__version__) >= LooseVersion("1.3"):
        datatype = torch.bool
    else:
        datatype = torch.uint8
    E = torch.eye(a.shape[-1], dtype=datatype).expand(*a.size())
    if LooseVersion(torch.__version__) >= LooseVersion("1.1"):
        E = E.type(torch.bool)
    return a[E].view(*a.size()[:-1]).sum(-1)


def allclose(
    a: Union[ComplexTensor, torch.Tensor],
    b: Union[ComplexTensor, torch.Tensor],
    rtol=1e-05,
    atol=1e-08,
    equal_nan=False,
) -> bool:
    if isinstance(a, ComplexTensor) and isinstance(b, ComplexTensor):
        return torch.allclose(
            a.real, b.real, rtol=rtol, atol=atol, equal_nan=equal_nan
        ) and torch.allclose(a.imag, b.imag, rtol=rtol, atol=atol, equal_nan=equal_nan)
    elif not isinstance(a, ComplexTensor) and isinstance(b, ComplexTensor):
        return torch.allclose(
            a.real, b.real, rtol=rtol, atol=atol, equal_nan=equal_nan
        ) and torch.allclose(
            torch.zeros_like(b.imag), b.imag, rtol=rtol, atol=atol, equal_nan=equal_nan
        )
    elif isinstance(a, ComplexTensor) and not isinstance(b, ComplexTensor):
        return torch.allclose(
            a.real, b, rtol=rtol, atol=atol, equal_nan=equal_nan
        ) and torch.allclose(
            a.imag, torch.zeros_like(a.imag), rtol=rtol, atol=atol, equal_nan=equal_nan
        )
    else:
        return torch.allclose(a, b, rtol=rtol, atol=atol, equal_nan=equal_nan)


def matmul(
    a: Union[ComplexTensor, torch.Tensor], b: Union[ComplexTensor, torch.Tensor]
) -> ComplexTensor:
    if isinstance(a, ComplexTensor) and isinstance(b, ComplexTensor):
        return a @ b
    elif not isinstance(a, ComplexTensor) and isinstance(b, ComplexTensor):
        o_real = torch.matmul(a, b.real)
        o_imag = torch.matmul(a, b.imag)
    elif isinstance(a, ComplexTensor) and not isinstance(b, ComplexTensor):
        return a @ b
    else:
        o_real = torch.matmul(a.real, b.real)
        o_imag = torch.zeros_like(o_real)
    return ComplexTensor(o_real, o_imag)


def solve(b: ComplexTensor, a: ComplexTensor) -> ComplexTensor:
    """Solve ax = b"""
    a = complex_matrix2real_matrix(a)
    b = complex_vector2real_vector(b)
    x, LU = torch.solve(b, a)
    return real_vector2complex_vector(x), real_matrix2complex_matrix(LU)


def complex_matrix2real_matrix(c: ComplexTensor) -> torch.Tensor:
    # NOTE(kamo):
    # Complex value can be expressed as follows
    #   a + bi => a * x + b y
    # where
    #   x = |1 0|  y = |0 -1|
    #       |0 1|,     |1  0|
    # A complex matrix can be also expressed as
    #   |A -B|
    #   |B  A|
    # and complex vector can be expressed as
    #   |A|
    #   |B|
    assert c.size(-2) == c.size(-1), c.size()
    # (∗, m, m) -> (*, 2m, 2m)
    return torch.cat(
        [torch.cat([c.real, -c.imag], dim=-1), torch.cat([c.imag, c.real], dim=-1)],
        dim=-2,
    )


def complex_vector2real_vector(c: ComplexTensor) -> torch.Tensor:
    # (∗, m, k) -> (*, 2m, k)
    return torch.cat([c.real, c.imag], dim=-2)


def real_matrix2complex_matrix(c: torch.Tensor) -> ComplexTensor:
    assert c.size(-2) == c.size(-1), c.size()
    # (∗, 2m, 2m) -> (*, m, m)
    n = c.size(-1)
    assert n % 2 == 0, n
    real = c[..., : n // 2, : n // 2]
    imag = c[..., n // 2 :, : n // 2]
    return ComplexTensor(real, imag)


def real_vector2complex_vector(c: torch.Tensor) -> ComplexTensor:
    # (∗, 2m, k) -> (*, m, k)
    n = c.size(-2)
    assert n % 2 == 0, n
    real = c[..., : n // 2, :]
    imag = c[..., n // 2 :, :]
    return ComplexTensor(real, imag)


##Data Loader Original

In [ ]:
import json
import os
import numpy as np
from random import shuffle
from torch.utils.data import Dataset, DataLoader
import random
import soundfile as sf
import librosa as lib
# from utils.utils import BatchInfo, pad_to_longest, logger_print


def _complex_from_ri(x_ri: torch.Tensor) -> torch.Tensor:
    # x_ri (..., 2) -> complex (...,)
    return torch.complex(x_ri[..., 0], x_ri[..., 1])

def _ri_from_complex(x_c: torch.Tensor) -> torch.Tensor:
    return torch.stack([x_c.real, x_c.imag], dim=-1)

class InstanceDataset(Dataset):
    def __init__(self,
                 mix_file_path,
                 bf_file_path,
                 target_file_path,
                 mix_json_path,
                 bf_json_path,
                 target_json_path,
                 batch_size,
                 is_check,
                 is_shuffle,
                 is_variance_norm,
                 is_chunk,
                 chunk_length,
                 sr,
                 ):
        super(InstanceDataset, self).__init__()
        self.mix_file_path = mix_file_path
        self.bf_file_path = bf_file_path
        self.target_file_path = target_file_path
        self.mix_json_path = mix_json_path
        self.bf_json_path = bf_json_path
        self.target_json_path = target_json_path
        self.batch_size = batch_size
        self.is_check = is_check
        self.is_shuffle = is_shuffle
        self.is_variance_norm = is_variance_norm
        self.is_chunk = is_chunk
        self.chunk_length = chunk_length
        self.sr = sr

        with open(mix_json_path, "r") as f:
            mix_json_list = json.load(f)
        with open(bf_json_path, "r") as f:
            bf_json_list = json.load(f)
        with open(target_json_path, "r") as f:
            target_json_list = json.load(f)

        # sort
        mix_json_list.sort()
        target_json_list.sort()
        bf_json_list.sort()

        if is_check:
            self.check_align(mix_json_list, target_json_list)
            self.check_align(mix_json_list, bf_json_list)

        if is_shuffle:
            random.seed(1234)  # fixed for reproducibility
            zipped_list = list(zip(mix_json_list, bf_json_list, target_json_list))
            shuffle(zipped_list)
            # the first type
            # mix_json_list, target_json_list = zip(*zipped_list)  # mix_json_list and target_json_list are tuple type
            # the second type
            json_list = list(map(list, zip(*zipped_list)))  # json_list (list:2)
            [mix_json_list, bf_json_list, target_json_list] = json_list

        mix_minibatch, bf_minibatch, target_minibatch = [], [], []
        start = 0
        while True:
            end = min(len(mix_json_list), start+batch_size)
            mix_minibatch.append(mix_json_list[start:end])
            bf_minibatch.append(bf_json_list[start:end])
            target_minibatch.append(target_json_list[start:end])
            start = end
            if end == len(mix_json_list):
                break
        self.mix_minibatch, self.bf_minibatch, self.target_minibatch = mix_minibatch, bf_minibatch, target_minibatch
        self.length = len(mix_minibatch)

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        mix_minibatch_list = self.mix_minibatch[index]
        bf_minibatch_list = self.bf_minibatch[index]
        target_minibatch_list = self.target_minibatch[index]
        mix_wav_list, bf_wav_list, target_wav_list, wav_len_list = [], [], [], []
        for id in range(len(mix_minibatch_list)):
            mix_filename = mix_minibatch_list[id]
            bf_filename = bf_minibatch_list[id]
            target_filename = target_minibatch_list[id]
            extracted_filename_from_mix = "_".join(mix_filename.split("_")[:-1])
            extracted_filename_from_target = "_".join(target_filename.split("_")[:-1])
            assert extracted_filename_from_mix == extracted_filename_from_target
            # read speech
            mix_wav, mix_sr = sf.read(os.path.join(self.mix_file_path, "{}.wav".format(mix_filename)))  # (L,M)
            bf_wav, bf_sr = sf.read(os.path.join(self.bf_file_path, "{}.wav".format(bf_filename)))
            target_wav, tar_sr = sf.read(os.path.join(self.target_file_path, "{}.wav".format(target_filename)))
            if mix_sr != self.sr or bf_sr != self.sr or tar_sr != self.sr:
                mix_wav, bf_wav, target_wav = lib.resample(mix_wav, mix_sr, self.sr), \
                                              lib.resample(bf_wav, bf_sr, self.sr),\
                                              lib.resample(target_wav, tar_sr, self.sr)
            if self.is_variance_norm:
                ref_mic = np.mean(mix_wav, axis=-1)   # mean mic is selected as the ref for normalization
                c = np.sqrt(len(ref_mic) / np.sum(ref_mic ** 2.0))
                mix_wav, bf_wav, target_wav = mix_wav * c, bf_wav * c, target_wav * c
            if self.is_chunk and (len(ref_mic) > int(self.sr*self.chunk_length)):
                wav_start = random.randint(0, len(ref_mic)-int(self.sr*self.chunk_length))
                mix_wav = mix_wav[wav_start:wav_start+int(self.sr*self.chunk_length), :]
                bf_wav = bf_wav[wav_start:wav_start+int(self.sr*self.chunk_length)]
                target_wav = target_wav[wav_start:wav_start+int(self.sr*self.chunk_length), :]
            mix_wav_list.append(mix_wav)
            bf_wav_list.append(bf_wav)
            target_wav_list.append(target_wav)
            wav_len_list.append(mix_wav.shape[0])
        return mix_wav_list, bf_wav_list, target_wav_list, wav_len_list

    @staticmethod
    def check_align(mix_list, target_list):
        logger_print("checking.................")
        is_ok = 1
        mix_error_list, target_error_list = [], []
        for i in range(len(mix_list)):
            extracted_filename_from_mix = "_".join(mix_list[i].split("_")[:-1])
            extracted_filename_from_target = "_".join(target_list[i].split("_")[:-1])
            if extracted_filename_from_mix != extracted_filename_from_target:
                is_ok = 0
                mix_error_list.append(extracted_filename_from_mix)
                target_error_list.append(extracted_filename_from_target)
        if is_ok == 0:
            for i in range(min(len(mix_error_list), len(target_error_list))):
                print("mix_file_name:{}, target_file_name:{}".format(mix_error_list[i],
                                                                     target_error_list[i]))
            raise Exception("Datasets between mix and target are not aligned!")
        else:
            logger_print("checking finished..............")

class InstanceDataloader(object):
    def __init__(self,
                 data_set,
                 num_workers,
                 pin_memory,
                 drop_last,
                 shuffle,
                 ):
        self.data_set = data_set
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.drop_last = drop_last
        self.shuffle = shuffle

        self.data_loader = DataLoader(dataset=data_set,
                                      num_workers=num_workers,
                                      pin_memory=pin_memory,
                                      drop_last=drop_last,
                                      shuffle=shuffle,
                                      collate_fn=self.collate_fn,
                                      batch_size=1
                                      )
    @staticmethod
    def collate_fn(batch):
        feats, bfs, labels, frame_mask_list = pad_to_longest(batch)
        return BatchInfo(feats, bfs, labels, frame_mask_list)

    def get_data_loader(self):
        return self.data_loader

## Model

In [ ]:
## model
import torch
import torch.nn as nn
from torch import Tensor
from torch.autograd import Variable
import math


class TaylorBeamformer(nn.Module):
    def __init__(self,
                 k1: list,
                 k2: list,
                 ref_mic: int,
                 c: int,
                 embed_dim: int,
                 fft_num: int,
                 order_num: int,
                 kd1: int,
                 cd1: int,
                 d_feat: int,
                 dilations: list,
                 group_num: int,
                 hid_node: int,
                 M: int,
                 rnn_type: str,
                 intra_connect: str,
                 inter_connect: str,
                 out_type: str,   # ["mask", "mapping"]
                 bf_type: str,  # ["embedding", "generalized", "mvdr"]
                 norm2d_type: str,  # ["BN", "IN"]
                 norm1d_type: str,
                 is_compress: bool,
                 is_total_separate: bool,  # whether the encoder in the spectral domain contains no spatial info
                 is_u2: bool,
                 is_1dgate: bool,
                 is_squeezed: bool,
                 is_causal: bool,
                 is_param_share: bool,
                 ):
        super(TaylorBeamformer, self).__init__()
        self.k1 = tuple(k1)
        self.k2 = tuple(k2)
        self.ref_mic = ref_mic
        self.c = c
        self.embed_dim = embed_dim
        self.fft_num = fft_num
        self.order_num = order_num
        self.kd1 = kd1
        self.cd1 = cd1
        self.d_feat = d_feat
        self.dilations = dilations
        self.group_num = group_num
        self.hid_node = hid_node
        self.M = M
        self.rnn_type = rnn_type
        self.intra_connect = intra_connect
        self.inter_connect = inter_connect
        self.out_type = out_type
        self.bf_type = bf_type
        self.norm2d_type = norm2d_type
        self.norm1d_type = norm1d_type
        self.is_compress = is_compress
        self.is_total_separate = is_total_separate
        self.is_u2 = is_u2
        self.is_1dgate = is_1dgate
        self.is_squeezed = is_squeezed
        self.is_causal = is_causal
        self.is_param_share = is_param_share

        #assert (out_type, bf_type) in [("mask", "mvdr"), ("mask", "generalized"), ("mapping", "embedding")]
        # Components
        self.zeroorderblock = ZeroOrderBlock(self.k1, self.k2, c, embed_dim, fft_num, kd1, cd1, d_feat, dilations,
                                             group_num, hid_node, M, rnn_type, intra_connect, inter_connect, out_type,
                                             bf_type, norm2d_type, norm1d_type, is_u2, is_1dgate, is_causal)
        if order_num > 0:
            if not is_total_separate:
                if is_u2:
                    self.highorderen = U2Net_Encoder(2*M, self.k1, self.k2, c, intra_connect, norm2d_type)
                else:
                    self.highorderen = UNet_Encoder(2*M, self.k1, c, norm2d_type)
            else:
                if is_u2:
                    self.highorderen = U2Net_Encoder(2, self.k1, self.k2, c, intra_connect, norm2d_type)
                else:
                    self.highorderen = UNet_Encoder(2, self.k1, c, norm2d_type)

            highorderblock_list = []
            if is_param_share:
                highorderblock_list.append(HighOrderBlock(kd1, cd1, d_feat, dilations, group_num, fft_num, is_1dgate,
                                                          is_causal, is_squeezed, norm1d_type))
            else:
                for i in range(order_num):
                    highorderblock_list.append(HighOrderBlock(kd1, cd1, d_feat, dilations, group_num, fft_num, is_1dgate,
                                                              is_causal, is_squeezed, norm1d_type))
            self.highorderblock_list = nn.ModuleList(highorderblock_list)

    def forward(self, inpt):
        """
        inpt: (B,T,F,M,2)
        return: spatial_x_wo_sum: (B,T,F,M,2) and out_term: (B,T,F,2)
        """
        if inpt.ndim == 4:
            inpt = inpt.unsqueeze(dim=-2)
        b_size, seq_len, freq_num, _, _ = inpt.shape
        # zero order process
        spatial_x = self.zeroorderblock(inpt)  # (B,T,F,2)
        # taylor unfolding process
        if self.is_compress:
            inpt_mag, inpt_phase = torch.norm(inpt, dim=-1)**0.5, torch.atan2(inpt[...,-1], inpt[...,0])
            inpt = torch.stack((inpt_mag*torch.cos(inpt_phase), inpt_mag*torch.sin(inpt_phase)), dim=-1)
            spatial_mag, spatial_phase = (torch.norm(spatial_x, dim=-1)+1e-10)**0.5, \
                                         torch.atan2(spatial_x[...,-1], spatial_x[...,0])
            spatial_x = torch.stack((spatial_mag*torch.cos(spatial_phase), spatial_mag*torch.sin(spatial_phase)), dim=1)
        else:
            spatial_x = spatial_x.permute(0,3,1,2).contiguous()
        out_term, pre_term = spatial_x, spatial_x  # (B,2,T,F)
        # high order encoding
        if self.order_num > 0:
            if not self.is_total_separate:
                inpt = inpt.contiguous().view(b_size, seq_len, freq_num, -1).permute(0,3,1,2).contiguous()  # (B,2M,T,F)
            else:
                inpt = inpt[...,self.ref_mic,:].permute(0,3,1,2).contiguous()   # (B,2,T,F)
            en_x, _ = self.highorderen(inpt)
            en_x = en_x.transpose(-2, -1).contiguous().view(b_size, -1, seq_len)

            for order_id in range(self.order_num):
                if self.is_param_share:
                    update_term = self.highorderblock_list[0](en_x, pre_term) + order_id * pre_term
                else:
                    update_term = self.highorderblock_list[order_id](en_x, pre_term) + order_id * pre_term
                pre_term = update_term
                out_term = out_term + update_term / math.factorial(order_id+1)
        return spatial_x.permute(0,2,3,1), out_term.permute(0,2,3,1)


class ZeroOrderBlock(nn.Module):
    def __init__(self,
                 k1: tuple,
                 k2: tuple,
                 c: int,
                 embed_dim: int,
                 fft_num: int,
                 kd1: int,
                 cd1: int,
                 d_feat: int,
                 dilations: list,
                 group_num: int,
                 hid_node: int,
                 M: int,
                 rnn_type: str,
                 intra_connect: str,
                 inter_connect: str,
                 out_type: str,
                 bf_type: str,
                 norm2d_type: str,
                 norm1d_type: str,
                 is_u2: bool,
                 is_1dgate: bool,
                 is_causal: bool,
                 ):
        super(ZeroOrderBlock, self).__init__()
        self.k1 = k1
        self.k2 = k2
        self.c = c
        self.embed_dim = embed_dim
        self.fft_num = fft_num
        self.kd1 = kd1
        self.cd1 = cd1
        self.d_feat = d_feat
        self.dilations = dilations
        self.group_num = group_num
        self.hid_node = hid_node
        self.M = M
        self.rnn_type = rnn_type
        self.intra_connect = intra_connect
        self.inter_connect = inter_connect
        self.out_type = out_type
        self.bf_type = bf_type
        self.norm2d_type = norm2d_type
        self.norm1d_type = norm1d_type
        self.is_u2 = is_u2
        self.is_1dgate = is_1dgate
        self.is_causal = is_causal
        # Components
        if is_u2:
            self.en = U2Net_Encoder(2*M, k1, k2, c, intra_connect, norm2d_type)
            self.de = U2Net_Decoder(c, k1, k2, embed_dim, fft_num, intra_connect, inter_connect, out_type, norm2d_type)
        else:
            self.en = UNet_Encoder(2*M, k1, c, norm2d_type)
            self.de = UNet_Decoder(c, k1, embed_dim, fft_num, inter_connect, out_type, norm2d_type)
        tcns = []
        for i in range(group_num):
            tcns.append(TCMGroup(kd1, cd1, d_feat, is_1dgate, dilations, is_causal, norm1d_type))
        self.tcns = nn.ModuleList(tcns)
        self.bf_module = BeamformingModule(embed_dim, M, hid_node, out_type, bf_type, rnn_type)


    def forward(self, inpt):
        """
        inpt: (B,T,F,M,2)
        return: (B,T,F,M,2)
        """
        b_size, seq_len, freq_num, channel_num, _ = inpt.shape
        inpt_x = inpt.contiguous().view(b_size, seq_len, freq_num, -1).permute(0,3,1,2)
        en_x, en_list = self.en(inpt_x)
        x = en_x.transpose(-2, -1).contiguous().view(b_size, -1, seq_len)
        x_acc = Variable(torch.zeros(x.size()), requires_grad=True).to(x.device)
        for i in range(self.group_num):
            x = self.tcns[i](x)
            x_acc += x
        x = x_acc
        x = x.view(b_size, -1, 4, seq_len).transpose(-2, -1).contiguous() # 4 denotes the freq size of the last encoding layer

        if self.out_type == "mask":
            est_s, est_n = self.de(inpt, x, en_list)
            bf_weight = self.bf_module(est_s, est_n)
        else:
            embed_x = self.de(inpt, x, en_list)
            bf_weight = self.bf_module(embed_x)
        bf_x = torch.sum(complex_mul(complex_conj(bf_weight), inpt), dim=-2)
        return bf_x


class HighOrderBlock(nn.Module):
    def __init__(self,
                 kd1: int,
                 cd1: int,
                 d_feat: int,
                 dilations: list,
                 group_num: int,
                 fft_num: int,
                 is_1dgate: bool,
                 is_causal: bool,
                 is_squeezed: bool,
                 norm1d_type: str,
                 ):
        super(HighOrderBlock, self).__init__()
        self.kd1 = kd1
        self.cd1 = cd1
        self.d_feat = d_feat
        self.dilations = dilations
        self.group_num = group_num
        self.fft_num = fft_num
        self.is_1dgate = is_1dgate
        self.is_causal = is_causal
        self.is_squeezed = is_squeezed
        self.norm1d_type = norm1d_type

        in_feat = (fft_num//2+1)*2 + d_feat
        self.in_conv = nn.Conv1d(in_feat, d_feat, 1)
        if not is_squeezed:
            tcm_r_list, tcm_i_list = [], []
            for i in range(group_num):
                tcm_r_list.append(TCMGroup(kd1, cd1, d_feat, is_1dgate, dilations, is_causal, norm1d_type))
                tcm_i_list.append(TCMGroup(kd1, cd1, d_feat, is_1dgate, dilations, is_causal, norm1d_type))
            self.tcms_r, self.tcms_i = nn.ModuleList(tcm_r_list), nn.ModuleList(tcm_i_list)
        else:
            tcm_list = []
            for i in range(group_num):
                tcm_list.append(TCMGroup(kd1, cd1, d_feat, is_1dgate, dilations, is_causal, norm1d_type))
            self.tcms = nn.ModuleList(tcm_list)
        self.real_resi, self.imag_resi = nn.Conv1d(d_feat, fft_num//2+1, 1), nn.Conv1d(d_feat, fft_num//2+1, 1)


    def forward(self, en_x: Tensor, pre_x: Tensor) -> Tensor:
        """
        :param en_x:  (B, C, T)
        :param pre_x: (B, 2, T, F)
        :return:  (B, 2, T, F)
        """
        assert en_x.ndim == 3 and pre_x.ndim == 4
        # fuse the features
        b_size, _, seq_len, freq_num = pre_x.shape
        x1 = pre_x.transpose(-2, -1).contiguous().view(b_size, -1, seq_len)
        x = torch.cat((en_x, x1), dim=1)
        # in conv
        x = self.in_conv(x)
        # STCMs
        if not self.is_squeezed:
            x_r, x_i = x, x
            for i in range(self.group_num):
                x_r, x_i = self.tcms_r[i](x_r), self.tcms_i[i](x_i)
        else:
            for i in range(self.group_num):
                x = self.tcms[i](x)
            x_r, x_i = x, x
        # generate real and imaginary parts
        x_r, x_i = self.real_resi(x_r).transpose(-2, -1), self.imag_resi(x_i).transpose(-2, -1)
        return torch.stack((x_r, x_i), dim=1).contiguous()


class UNet_Encoder(nn.Module):
    def __init__(self,
                 cin: int,
                 k1: tuple,
                 c: int,
                 norm2d_type: str,
                 ):
        super(UNet_Encoder, self).__init__()
        self.cin = cin
        self.k1 = k1
        self.c = c
        self.norm2d_type = norm2d_type
        kernel_begin = (k1[0], 5)
        stride = (1, 2)
        c_final = 64
        unet = []
        unet.append(nn.Sequential(
            GateConv2d(cin, c, kernel_begin, stride, padding=(0, 0, k1[0]-1, 0)),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConv2d(c, c, k1, stride, padding=(0, 0, k1[0]-1, 0)),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConv2d(c, c, k1, stride, padding=(0, 0, k1[0]-1, 0)),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConv2d(c, c, k1, stride, padding=(0, 0, k1[0]-1, 0)),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConv2d(c, c_final, k1, (1,2), padding=(0, 0, k1[0]-1, 0)),
            NormSwitch(norm2d_type, "2D", c_final),
            nn.PReLU(c_final)))
        self.unet_list = nn.ModuleList(unet)

    def forward(self, x: Tensor) -> tuple:
        en_list = []
        for i in range(len(self.unet_list)):
            x = self.unet_list[i](x)
            en_list.append(x)
        return x, en_list


class UNet_Decoder(nn.Module):
    def __init__(self,
                 c: int,
                 k1: tuple,
                 embed_dim: int,
                 fft_num: int,
                 inter_connect: str,
                 out_type: str,
                 norm2d_type: str,
                 ):
        super(UNet_Decoder, self).__init__()
        self.k1 = k1
        self.c = c
        self.embed_dim = embed_dim
        self.fft_num = fft_num
        self.inter_connect = inter_connect
        self.out_type = out_type
        self.norm2d_type = norm2d_type

        kernel_end = (k1[0], 5)
        stride = (1, 2)
        unet = []
        if inter_connect == "add":
            inter_c = c
            c_begin = 64
        elif inter_connect == "cat":
            inter_c = c * 2
            c_begin = 64 * 2
        else:
            raise RuntimeError("Skip connections only support add or concatenate operation")

        unet.append(nn.Sequential(
            GateConvTranspose2d(c_begin, c, k1, stride),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConvTranspose2d(inter_c, c, k1, stride),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConvTranspose2d(inter_c, c, k1, stride),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        unet.append(nn.Sequential(
            GateConvTranspose2d(inter_c, c, k1, stride),
            NormSwitch(norm2d_type, "2D", c),
            nn.PReLU(c)))
        self.unet_list = nn.ModuleList(unet)
        if out_type == "mask":
            self.conv = nn.Sequential(
                GateConvTranspose2d(inter_c, c, kernel_end, stride),
                NormSwitch(norm2d_type, "2D", c),
                nn.PReLU(c)
            )
            self.mask_s = nn.Sequential(
                nn.Conv2d(c, 2, (1, 1), (1, 1)),
                nn.Linear(fft_num//2+1, fft_num//2+1)
            )
            self.mask_n = nn.Sequential(
                nn.Conv2d(c, 2, (1, 1), (1, 1)),
                nn.Linear(fft_num//2+1, fft_num//2+1)
            )
        elif out_type == "mapping":
            self.embed = nn.Sequential(
                GateConvTranspose2d(inter_c, embed_dim, kernel_end, stride),
                nn.Linear(fft_num//2+1, fft_num//2+1)
            )

    def forward(self, inpt: Tensor, x: Tensor, en_list: list):
        """
        inpt: (B,T,F,M,2)
        return: (B,-1,T,F)
        """
        b_size, seq_len, freq_num, _, _ = inpt.shape
        if self.inter_connect == "add":
            for i in range(len(self.unet_list)):
                tmp = x + en_list[-(i + 1)]
                x = self.unet_list[i](tmp)
            x = x + en_list[0]
        elif self.inter_connect == "cat":
            for i in range(len(self.unet_list)):
                tmp = torch.cat((x, en_list[-(i + 1)]), dim=1)
                x = self.unet_list[i](tmp)
            x = torch.cat((x, en_list[0]), dim=1)
        else:
            raise RuntimeError("only add and cat are supported")
        # output
        if self.out_type == "mask":
            x = self.conv(x)
            mask_s, mask_n = self.mask_s(x).permute(0,2,3,1).contiguous().unsqueeze(dim=-2), \
                             self.mask_n(x).permute(0,2,3,1).contiguous().unsqueeze(dim=-2)
            est_s, est_n = complex_mul(inpt, mask_s), complex_mul(inpt, mask_n)
            return est_s, est_n
        elif self.out_type == "mapping":
            out_x = self.embed(x).permute(0,2,3,1).contiguous()
            return out_x
        else:
            raise RuntimeError("only mask and mapping are supported")


class U2Net_Encoder(nn.Module):
    def __init__(self,
                 cin: int,
                 k1: tuple,
                 k2: tuple,
                 c: int,
                 intra_connect: str,
                 norm2d_type: str,
                 ):
        super(U2Net_Encoder, self).__init__()
        self.cin = cin
        self.k1 = k1
        self.k2 = k2
        self.c = c
        self.intra_connect = intra_connect
        self.norm2d_type = norm2d_type
        c_last = 64
        kernel_begin = (k1[0], 5)
        stride = (1, 2)
        meta_unet = []
        meta_unet.append(
            En_unet_module(cin, c, kernel_begin, k2, intra_connect, norm2d_type, scale=4, de_flag=False))
        meta_unet.append(
            En_unet_module(c, c, k1, k2, intra_connect, norm2d_type, scale=3, de_flag=False))
        meta_unet.append(
            En_unet_module(c, c, k1, k2, intra_connect, norm2d_type, scale=2, de_flag=False))
        meta_unet.append(
            En_unet_module(c, c, k1, k2, intra_connect, norm2d_type, scale=1, de_flag=False))
        self.meta_unet_list = nn.ModuleList(meta_unet)
        self.last_conv = nn.Sequential(
            GateConv2d(c, c_last, k1, stride, (0, 0, k1[0]-1, 0)),
            NormSwitch(norm2d_type, "2D", c_last),
            nn.PReLU(c_last)
        )

    def forward(self, x: Tensor) -> tuple:
        en_list = []
        for i in range(len(self.meta_unet_list)):
            x = self.meta_unet_list[i](x)
            en_list.append(x)
        x = self.last_conv(x)
        en_list.append(x)
        return x, en_list


class U2Net_Decoder(nn.Module):
    def __init__(self,
                 c: int,
                 k1: tuple,
                 k2: tuple,
                 embed_dim: int,
                 fft_num: int,
                 intra_connect: str,
                 inter_connect: str,
                 out_type: str,
                 norm2d_type: str,
                 ):
        super(U2Net_Decoder, self).__init__()
        self.c = c
        self.k1 = k1
        self.k2 = k2
        self.embed_dim = embed_dim
        self.fft_num = fft_num
        self.intra_connect = intra_connect
        self.inter_connect = inter_connect
        self.out_type = out_type
        self.norm2d_type = norm2d_type

        kernel_end = (k1[0], 5)
        stride = (1, 2)
        meta_unet = []
        if inter_connect == "add":
            inter_c = c
            c_begin = 64
        elif inter_connect == "cat":
            inter_c = c*2
            c_begin = 64*2
        else:
            raise RuntimeError("Skip connections only support add or concatenate operation")
        meta_unet.append(
            En_unet_module(c_begin, c, k1, k2, intra_connect, norm2d_type, scale=1, de_flag=True))
        meta_unet.append(
            En_unet_module(inter_c, c, k1, k2, intra_connect, norm2d_type, scale=2, de_flag=True))
        meta_unet.append(
            En_unet_module(inter_c, c, k1, k2, intra_connect, norm2d_type, scale=3, de_flag=True))
        meta_unet.append(
            En_unet_module(inter_c, c, k1, k2, intra_connect, norm2d_type, scale=4, de_flag=True))
        self.meta_unet_list = nn.ModuleList(meta_unet)
        if out_type == "mask":
            self.conv = nn.Sequential(
                GateConvTranspose2d(inter_c, c, kernel_end, stride),
                NormSwitch(norm2d_type, "2D", c),
                nn.PReLU(c)
            )
            self.mask_s = nn.Sequential(
                nn.Conv2d(c, 2, (1, 1), (1, 1)),
                nn.Linear(fft_num//2+1, fft_num//2+1)
            )
            self.mask_n = nn.Sequential(
                nn.Conv2d(c, 2, (1, 1), (1, 1)),
                nn.Linear(fft_num//2+1, fft_num//2+1)
            )
        elif out_type == "mapping":
            self.embed = nn.Sequential(
                GateConvTranspose2d(inter_c, embed_dim, kernel_end, stride),
                nn.Linear(fft_num//2+1, fft_num//2+1)
            )

    def forward(self, inpt: Tensor, x: Tensor, en_list: list):
        """
        inpt: (B,T,F,M,2)
        return: (B,T,F,M,2) or (B,T,F,K)
        """
        b_size, seq_len, freq_num, _, _ = inpt.shape
        if self.inter_connect == "add":
            for i in range(len(self.meta_unet_list)):
                tmp = x + en_list[-(i+1)]
                x = self.meta_unet_list[i](tmp)
            x = x + en_list[0]
        elif self.inter_connect == "cat":
            for i in range(len(self.meta_unet_list)):
                tmp = torch.cat((x, en_list[-(i+1)]), dim=1)
                x = self.meta_unet_list[i](tmp)
            x = torch.cat((x, en_list[0]), dim=1)
        else:
            raise RuntimeError("only add and cat are supported")
        # output
        if self.out_type == "mask":
            x = self.conv(x)
            mask_s, mask_n = self.mask_s(x).permute(0, 2, 3, 1).contiguous().unsqueeze(dim=-2), \
                             self.mask_n(x).permute(0, 2, 3, 1).contiguous().unsqueeze(dim=-2)
            est_s, est_n = complex_mul(inpt, mask_s), complex_mul(inpt, mask_n)
            return est_s, est_n
        elif self.out_type == "mapping":
            out_x = self.embed(x).permute(0, 2, 3, 1).contiguous()
            return out_x
        else:
            raise RuntimeError("only mask and mapping are supported")


class En_unet_module(nn.Module):
    def __init__(self,
                 cin: int,
                 cout: int,
                 k1: tuple,
                 k2: tuple,
                 intra_connect: str,
                 norm2d_type: str,
                 scale: int,
                 de_flag: bool = False,
                 ):
        super(En_unet_module, self).__init__()
        self.cin = cin
        self.cout = cout
        self.k1 = k1
        self.k2 = k2
        self.intra_connect = intra_connect
        self.norm2d_type = norm2d_type
        self.scale = scale
        self.de_flag = de_flag

        in_conv_list = []
        if de_flag is False:
            in_conv_list.append(GateConv2d(cin, cout, k1, (1, 2), (0, 0, k1[0]-1, 0)))
        else:
            in_conv_list.append(GateConvTranspose2d(cin, cout, k1, (1, 2)))
        in_conv_list.append(NormSwitch(norm2d_type, "2D", cout))
        in_conv_list.append(nn.PReLU(cout))
        self.in_conv = nn.Sequential(*in_conv_list)

        enco_list, deco_list = [], []
        for _ in range(scale):
            enco_list.append(Conv2dunit(k2, cout, norm2d_type))
        for i in range(scale):
            if i == 0:
                deco_list.append(Deconv2dunit(k2, cout, "add", norm2d_type))
            else:
                deco_list.append(Deconv2dunit(k2, cout, intra_connect, norm2d_type))
        self.enco = nn.ModuleList(enco_list)
        self.deco = nn.ModuleList(deco_list)
        self.skip_connect = Skip_connect(intra_connect)


    def forward(self, inputs: Tensor) -> Tensor:
        x_resi = self.in_conv(inputs)
        x = x_resi
        x_list = []
        for i in range(len(self.enco)):
            x = self.enco[i](x)
            x_list.append(x)

        for i in range(len(self.deco)):
            if i == 0:
                x = self.deco[i](x)
            else:
                x_con = self.skip_connect(x, x_list[-(i+1)])
                x = self.deco[i](x_con)
        x_resi = x_resi + x
        del x_list
        return x_resi


class Conv2dunit(nn.Module):
    def __init__(self,
                 k: tuple,
                 c: int,
                 norm2d_type: str,
                 ):
        super(Conv2dunit, self).__init__()
        self.k, self.c = k, c
        self.norm2d_type = norm2d_type
        k_t = k[0]
        stride = (1, 2)
        if k_t > 1:
            self.conv = nn.Sequential(
                nn.ConstantPad2d((0, 0, k_t-1, 0), value=0.),
                nn.Conv2d(c, c, k, stride),
                NormSwitch(norm2d_type, "2D", c),
                nn.PReLU(c)
                )
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(c, c, k, stride),
                NormSwitch(norm2d_type, "2D", c),
                nn.PReLU(c)
            )

    def forward(self, inputs: Tensor) -> Tensor:
        return self.conv(inputs)


class Deconv2dunit(nn.Module):
    def __init__(self,
                 k: tuple,
                 c: int,
                 intra_connect: str,
                 norm2d_type: str,
                 ):
        super(Deconv2dunit, self).__init__()
        self.k, self.c = k, c
        self.intra_connect = intra_connect
        self.norm2d_type = norm2d_type
        k_t = k[0]
        stride = (1, 2)
        deconv_list = []
        if self.intra_connect == "add":
            if k_t > 1:
                deconv_list.append(nn.ConvTranspose2d(c, c, k, stride)),
                deconv_list.append(Chomp_T(k_t-1))
            else:
                deconv_list.append(nn.ConvTranspose2d(c, c, k, stride))
        elif self.intra_connect == "cat":
            if k_t > 1:
                deconv_list.append(nn.ConvTranspose2d(2*c, c, k, stride))
                deconv_list.append(Chomp_T(k_t-1))
            else:
                deconv_list.append(nn.ConvTranspose2d(2*c, c, k, stride))
        deconv_list.append(NormSwitch(norm2d_type, "2D", c))
        deconv_list.append(nn.PReLU(c))
        self.deconv = nn.Sequential(*deconv_list)

    def forward(self, inputs: Tensor) -> Tensor:
        assert inputs.dim() == 4
        return self.deconv(inputs)


class GateConv2d(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size: tuple,
                 stride: tuple,
                 padding: tuple,
                 ):
        super(GateConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        k_t = kernel_size[0]
        if k_t > 1:
            self.conv = nn.Sequential(
                nn.ConstantPad2d(padding, value=0.),
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels*2, kernel_size=kernel_size, stride=stride))
        else:
            self.conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels*2, kernel_size=kernel_size,
                                  stride=stride)
    def forward(self, inputs: Tensor) -> Tensor:
        if inputs.dim() == 3:
            inputs = inputs.unsqueeze(dim=1)
        x = self.conv(inputs)
        outputs, gate = x.chunk(2, dim=1)
        return outputs * gate.sigmoid()


class GateConvTranspose2d(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size: tuple,
                 stride: tuple,
                 ):
        super(GateConvTranspose2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride

        k_t = kernel_size[0]
        if k_t > 1:
            self.conv = nn.Sequential(
                nn.ConvTranspose2d(in_channels=in_channels, out_channels=out_channels*2, kernel_size=kernel_size,
                                   stride=stride),
                Chomp_T(k_t-1))
        else:
            self.conv = nn.ConvTranspose2d(in_channels=in_channels, out_channels=out_channels*2, kernel_size=kernel_size,
                                           stride=stride)

    def forward(self, inputs: Tensor) -> Tensor:
        assert inputs.dim() == 4
        x = self.conv(inputs)
        outputs, gate = x.chunk(2, dim=1)
        return outputs * gate.sigmoid()


class Skip_connect(nn.Module):
    def __init__(self,
                 connect):
        super(Skip_connect, self).__init__()
        self.connect = connect

    def forward(self, x_main, x_aux):
        if self.connect == "add":
            x = x_main + x_aux
        elif self.connect == "cat":
            x = torch.cat((x_main, x_aux), dim=1)
        return x


class TCMGroup(nn.Module):
    def __init__(self,
                 kd1: int,
                 cd1: int,
                 d_feat: int,
                 is_gate: bool,
                 dilations: list,
                 is_causal: bool,
                 norm1d_type: str,
                 ):
        super(TCMGroup, self).__init__()
        self.kd1 = kd1
        self.cd1 = cd1
        self.d_feat = d_feat
        self.is_gate = is_gate
        self.dilations = dilations
        self.is_causal = is_causal
        self.norm1d_type = norm1d_type

        tcm_list = []
        for i in range(len(dilations)):
            tcm_list.append(SqueezedTCM(kd1, cd1, dilation=dilations[i], d_feat=d_feat, is_gate=is_gate,
                                        is_causal=is_causal, norm1d_type=norm1d_type))
        self.tcm_list = nn.ModuleList(tcm_list)

    def forward(self, inputs: Tensor) -> Tensor:
        x = inputs
        for i in range(len(self.dilations)):
            x = self.tcm_list[i](x)
        return x


class SqueezedTCM(nn.Module):
    def __init__(self,
                 kd1: int,
                 cd1: int,
                 dilation: int,
                 d_feat: int,
                 is_gate: bool,
                 is_causal: bool,
                 norm1d_type: str,
                 ):
        super(SqueezedTCM, self).__init__()
        self.kd1 = kd1
        self.cd1 = cd1
        self.dilation = dilation
        self.d_feat = d_feat
        self.is_gate = is_gate
        self.is_causal = is_causal
        self.norm1d_type = norm1d_type

        self.in_conv = nn.Conv1d(d_feat, cd1, kernel_size=1, bias=False)
        if is_causal:
            pad = ((kd1-1)*dilation, 0)
        else:
            pad = ((kd1-1)*dilation//2, (kd1-1)*dilation//2)
        self.left_conv = nn.Sequential(
            nn.PReLU(cd1),
            NormSwitch(norm1d_type, "1D", cd1),
            nn.ConstantPad1d(pad, value=0.),
            nn.Conv1d(cd1, cd1, kernel_size=kd1, dilation=dilation, bias=False)
        )
        if is_gate:
            self.right_conv = nn.Sequential(
                nn.PReLU(cd1),
                NormSwitch(norm1d_type, "1D", cd1),
                nn.ConstantPad1d(pad, value=0.),
                nn.Conv1d(cd1, cd1, kernel_size=kd1, dilation=dilation, bias=False),
                nn.Sigmoid()
            )
        self.out_conv = nn.Sequential(
            nn.PReLU(cd1),
            NormSwitch(norm1d_type, "1D", cd1),
            nn.Conv1d(cd1, d_feat, kernel_size=1, bias=False)
        )

    def forward(self, inputs: Tensor) -> Tensor:
        resi = inputs
        x = self.in_conv(inputs)
        if self.is_gate:
            x = self.left_conv(x) * self.right_conv(x)
        else:
            x = self.left_conv(x)
        x = self.out_conv(x)
        x = x + resi
        return x

class BeamformingModule(nn.Module):
    def __init__(self,
                 embed_dim: int,
                 M: int,
                 hid_node: int,
                 out_type: str,
                 bf_type: str,
                 rnn_type: str,
                 ):
        super(BeamformingModule, self).__init__()
        self.embed_dim = embed_dim
        self.M = M
        self.hid_node = hid_node
        self.out_type = out_type
        self.bf_type = bf_type
        self.rnn_type = rnn_type
        assert out_type in ["mask", "mapping"]
        assert bf_type in ["embedding", "generalized", "mvdr"]
        if out_type == "mask":
            inpt_dim = 2*2*M*M
        elif out_type == "mapping":
            inpt_dim = embed_dim
        else:
            raise RuntimeError("only mask and mapping are supported")

        if bf_type in ["embedding", "generalized"]:
            self.norm = nn.LayerNorm([inpt_dim])
            self.rnn = getattr(nn, rnn_type)(input_size=inpt_dim, hidden_size=hid_node, num_layers=2, batch_first=True)
            self.w_dnn = nn.Sequential(
                nn.Linear(hid_node, hid_node),
                nn.ReLU(True),
                nn.Linear(hid_node, 2*M)
            )
        elif bf_type == "mvdr":
            self.norm1 = nn.LayerNorm([inpt_dim//2])
            self.norm2 = nn.LayerNorm([inpt_dim//2])
            self.rnn1 = getattr(nn, rnn_type)(input_size=inpt_dim//2, hidden_size=hid_node, num_layers=2, batch_first=True)
            self.rnn2 = getattr(nn, rnn_type)(input_size=inpt_dim//2, hidden_size=hid_node, num_layers=2, batch_first=True)
            self.pca_dnn = nn.Sequential(
                nn.Linear(hid_node, hid_node),
                nn.ReLU(True),
                nn.Linear(hid_node, 2*M)
            )
            self.inverse_dnn = nn.Sequential(
                nn.Linear(hid_node, hid_node),
                nn.ReLU(True),
                nn.Linear(hid_node, 2*M*M)
            )

    def forward(self, inpt1, inpt2=None):
        if self.out_type == "mask":
            est_s, est_n = inpt1, inpt2
            complex_s = ComplexTensor(est_s[...,0], est_s[...,-1])  # (B,T,F,M)
            complex_n = ComplexTensor(est_n[...,0], est_n[...,-1])  # (B,T,F,M)
            cov_s = F.einsum("...m,...n->...mn", [complex_s.conj(), complex_s])  # (B,T,F,M,M)
            cov_n = F.einsum("...m,...n->...mn", [complex_n.conj(), complex_n])  # (B,T,F,M,M)
            b_size, seq_len, freq_num, M, M = cov_s.shape
            cov_s, cov_n = cov_s.view(b_size, seq_len, freq_num, -1), cov_n.view(b_size, seq_len, freq_num, -1)
            cov_ss = torch.cat((cov_s.real, cov_s.imag), dim=-1).permute(0,3,1,2)  # (B,2*M*M,T,F)
            cov_nn = torch.cat((cov_n.real, cov_n.imag), dim=-1).permute(0,3,1,2)  # (B,2*M*M,T,F)
        else:
            embed_x = inpt1.permute(0,3,1,2)  # (B,-1,T,F)
            b_size, _, seq_len, freq_num = embed_x.shape

        if self.bf_type == "mvdr":
            cov_ss, cov_nn = self.norm1(cov_ss.permute(0,3,2,1).contiguous()), \
                             self.norm2(cov_nn.permute(0,3,2,1).contiguous())
            cov_ss, cov_nn = cov_ss.view(b_size*freq_num, seq_len, -1), \
                             cov_nn.view(b_size*freq_num, seq_len, -1)
            # steer vestor
            h1, _ = self.rnn1(cov_ss)
            steer_vec = self.pca_dnn(h1)
            steer_vec = steer_vec.view(b_size, freq_num, seq_len, self.M, 2).transpose(1,2)  # (B,T,F,M,2)
            # inverse rnn
            h2, _ = self.rnn2(cov_nn)
            inverse_phi = self.inverse_dnn(h2)
            inverse_phi = inverse_phi.view(b_size, freq_num, seq_len, self.M, self.M, 2).transpose(1, 2) # (B,T,F,M,M,2)
            # mvdr
            complex_steer_vec = ComplexTensor(steer_vec[...,0], steer_vec[...,-1])  # (B,T,F,M)
            complex_inverse_phi = ComplexTensor(inverse_phi[...,0], inverse_phi[...,-1])  # (B,T,F,M,M)
            nomin = F.einsum("...mn,...n->...m", [complex_inverse_phi, complex_steer_vec])  # (B,T,F,M)
            denomin = F.einsum("...m,...m->...", [complex_steer_vec.conj(), nomin])   # (B,T,F)
            bf_weight = nomin / denomin.unsqueeze(dim=-1)
            bf_weight = torch.stack((bf_weight.real, bf_weight.imag), dim=-1)   # (B,T,F,M,2)
        elif self.bf_type == "generalized":
            x = self.norm(torch.cat((cov_ss, cov_nn), dim=1).permute(0,3,2,1).contiguous())
            x = x.view(b_size*freq_num, seq_len, -1)
            h, _ = self.rnn(x)
            bf_weight = self.w_dnn(h)
            bf_weight = bf_weight.view(b_size, freq_num, seq_len, self.M, 2).transpose(1, 2)
        elif self.bf_type == "embedding":
            x = self.norm(embed_x.permute(0,3,2,1).contiguous())
            x = x.view(b_size*freq_num, seq_len, -1)
            h, _ = self.rnn(x)
            bf_weight = self.w_dnn(h)
            bf_weight = bf_weight.view(b_size, freq_num, seq_len, self.M, 2).transpose(1, 2)
        else:
            raise Exception("only mvdr, generalized, and embedding are supported")
        return bf_weight


class Chomp_T(nn.Module):
    def __init__(self,
                 t: int):
        super(Chomp_T, self).__init__()
        self.t = t

    def forward(self, x):
        return x[:, :, :-self.t, :]



if __name__ == "__main__":
    net = TaylorBeamformer(
        k1=[1,3],
        k2=[2,3],
        ref_mic=0,
        c=64,
        embed_dim=64,
        fft_num=320,
        order_num=4,
        kd1=5,
        cd1=64,
        d_feat=256,
        dilations=[1,2,5,9],
        group_num=2,
        hid_node=64,
        M=6,
        rnn_type="LSTM",
        intra_connect="cat",
        inter_connect="cat",
        out_type="mapping",
        bf_type="embedding",
        norm2d_type="BN",
        norm1d_type="BN",
        is_compress=True,
        is_total_separate=False,
        is_u2=True,
        is_1dgate=True,
        is_squeezed=True,
        is_causal=True,
        is_param_share=False
    ).cuda()
    x = torch.rand([3,31,161,6,2]).cuda()

## Train

In [ ]:
## solver - train
import numpy as np
import os
import torch
import torch.nn as nn
import time
import importlib
import hdf5storage

train_epoch, val_epoch, val_metric_epoch = [], [], []  # for loss, loss and metric score
# from torch.utils.tensorboard import SummaryWriter


class Solver(object):
    def __init__(self,
                 data,
                 net,
                 optimizer,
                 save_name_dict,
                 args,
                 ):
        self.train_dataloader = data["train_loader"]
        self.val_dataloader = data["val_loader"]
        self.net = net
        # optimizer part
        self.optimizer = optimizer
        self.lr = args["optimizer"]["lr"]
        self.gradient_norm = args["optimizer"]["gradient_norm"]
        self.epochs = args["optimizer"]["epochs"]
        self.halve_lr = args["optimizer"]["halve_lr"]
        self.early_stop = args["optimizer"]["early_stop"]
        self.halve_freq = args["optimizer"]["halve_freq"]
        self.early_stop_freq = args["optimizer"]["early_stop_freq"]
        self.print_freq = args["optimizer"]["print_freq"]
        self.metric_options = args["optimizer"]["metric_options"]
        # loss part
        self.loss_path = args["loss_function"]["path"]
        self.spectral_loss = args["loss_function"]["spectral"]["classname"]
        self.spatial_weight = args["loss_function"]["spatial_weight"]
        self.spectral_weight = args["loss_function"]["spectral_weight"]
        self.alpha = args["loss_function"]["alpha"]
        self.l_type = args["loss_function"]["l_type"]
        # signal part
        self.sr = args["signal"]["sr"]
        self.win_size = args["signal"]["win_size"]
        self.win_shift = args["signal"]["win_shift"]
        self.fft_num = args["signal"]["fft_num"]
        self.is_compress = args["signal"]["is_compress"]
        self.ref_mic = args["signal"]["ref_mic"]
        # path part
        self.is_checkpoint = args["path"]["is_checkpoint"]
        self.is_resume_reload = args["path"]["is_resume_reload"]
        self.checkpoint_load_path = args["path"]["checkpoint_load_path"]
        self.checkpoint_load_filename = args["path"]["checkpoint_load_filename"]
        self.loss_save_path = args["path"]["loss_save_path"]
        self.model_best_path = args["path"]["model_best_path"]
        # sava name
        self.loss_save_filename = save_name_dict["loss_filename"]
        self.best_model_save_filename = save_name_dict["best_model_filename"]
        self.checkpoint_save_filename = save_name_dict["checkpoint_filename"]

        self.train_loss = torch.Tensor(self.epochs)
        self.val_loss = torch.Tensor(self.epochs)
        # set loss funcs
        self.spatial_loss = SpatialFilterLoss(self.alpha, self.l_type)
        self.spectral_loss = ComMagEuclideanLoss(self.alpha, self.l_type)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self._reset()

        # summarywriter
        # self.tensorboard_path = "./" + args["path"]["logging_path"] + "/" + args["save"]["tensorboard_filename"]
        # if not os.path.exists(self.tensorboard_path):
        #     os.makedirs(self.tensorboard_path)
        #self.writer = SummaryWriter(self.tensorboard_path, max_queue=5, flush_secs=30)

    def _reset(self):
        # Reset
        if self.is_resume_reload:
            checkpoint = torch.load(os.path.join(self.checkpoint_load_path, self.checkpoint_load_filename), weights_only=False)
            self.net.load_state_dict(checkpoint["model_state_dict"])
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            for state in self.optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor):
                        state[k] = v.to(self.device)
            self.start_epoch = checkpoint["start_epoch"]
            self.prev_val_loss = checkpoint["val_loss"]  # val loss
            self.prev_val_metric = checkpoint["val_metric"]
            self.best_val_metric = checkpoint["best_val_metric"]
            self.val_no_impv = checkpoint["val_no_impv"]
            self.halving = checkpoint["halving"]
        else:
            self.start_epoch = 0
            self.prev_val_loss = float("inf")
            self.prev_val_metric = -float("inf")
            self.best_val_metric = -float("inf")
            self.val_no_impv = 0
            self.halving = False

    def train(self):
        logger_print("Begin to train....")
        self.net.to(self.device)
        for epoch in range(self.start_epoch, self.epochs):
            begin_time = time.time()
            # training phase
            logger_print("-" * 90)
            start_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            logger_print(f"Epoch id:{int(epoch + 1)}, Training phase, Start time:{start_time}")
            self.net.train()
            train_avg_loss = self._run_one_epoch(epoch, val_opt=False)
            # self.writer.add_scalar(f"Loss/Training_Loss", train_avg_loss, epoch)
            end_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            logger_print(f"Epoch if:{int(epoch + 1)}, Training phase, End time:{end_time}, "
                         f"Training loss:{train_avg_loss}")

            # Cross val
            start_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            logger_print(f"Epoch id:{int(epoch + 1)}, Validation phase, Start time:{start_time}")
            self.net.eval()    # norm and dropout is off
            val_avg_loss, val_avg_metric = self._run_one_epoch(epoch, val_opt=True)
            # self.writer.add_scalar(f"Loss/Validation_Loss", val_avg_loss, epoch)
            # self.writer.add_scalar(f"Loss/Validation_Metric", val_avg_metric, epoch)
            end_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            logger_print(f"Epoch if:{int(epoch + 1)}, Validation phase, End time:{end_time}, "
                         f"Validation loss:{val_avg_loss}, Validation metric score:{val_avg_metric}")
            end_time = time.time()
            print(f"{end_time-begin_time}s in {epoch+1}th epoch")
            logger_print("-" * 90)

            # whether to save checkpoint at current epoch
            if self.is_checkpoint:
                cpk_dic = {}
                cpk_dic["model_state_dict"] = self.net.state_dict()
                cpk_dic["optimizer_state_dict"] = self.optimizer.state_dict()
                cpk_dic["train_loss"] = train_avg_loss
                cpk_dic["val_loss"] = val_avg_loss
                cpk_dic["val_metric"] = val_avg_metric
                cpk_dic["best_val_metric"] = self.best_val_metric
                cpk_dic["start_epoch"] = epoch+1
                cpk_dic["val_no_impv"] = self.val_no_impv
                cpk_dic["halving"] = self.halving
                torch.save(cpk_dic, os.path.join(self.checkpoint_load_path, "Epoch_{}_{}_{}".format(epoch+1,
                                                        self.net.__class__.__name__, self.checkpoint_save_filename)))
            # record loss
            # self.train_loss[epoch] = train_avg_loss
            # self.val_loss[epoch] = val_avg_loss

            train_epoch.append(train_avg_loss)
            val_epoch.append(val_avg_loss)
            val_metric_epoch.append(val_avg_metric)

            # save loss
            loss = {}
            loss["train_loss"] = train_epoch
            loss["val_loss"] = val_epoch
            loss["val_metric"] = val_metric_epoch

            if not self.is_resume_reload:
                hdf5storage.savemat(os.path.join(self.loss_save_path, self.loss_save_filename), loss)
            else:
                hdf5storage.savemat(os.path.join(self.loss_save_path, "resume_cpk_{}".format(self.loss_save_filename)),
                                    loss)

            # lr halve and Early stop
            if self.halve_lr:
                if val_avg_metric <= self.prev_val_metric:
                    self.val_no_impv += 1
                    if self.val_no_impv == self.halve_freq:
                        self.halving = True
                    if (self.val_no_impv >= self.early_stop_freq) and self.early_stop:
                        logger_print("No improvements and apply early-stopping")
                        break
                else:
                    self.val_no_impv = 0

            if self.halving:
                optim_state = self.optimizer.state_dict()
                optim_state["param_groups"][0]["lr"] = optim_state["param_groups"][0]["lr"] / 2.0
                self.optimizer.load_state_dict(optim_state)
                logger_print("Learning rate is adjusted to %5f" % (optim_state["param_groups"][0]["lr"]))
                self.halving = False
            self.prev_val_metric = val_avg_metric

            if val_avg_metric > self.best_val_metric:
                self.best_val_metric = val_avg_metric
                torch.save(self.net.state_dict(), os.path.join(self.model_best_path, self.best_model_save_filename))
                logger_print(f"Find better model, saving to {self.best_model_save_filename}")
            else:
                logger_print("Did not find better model")


    @torch.no_grad()
    def _val_batch(self, batch_info):
        batch_mix_wav = batch_info.feats.to(self.device)  # (B,L,M)
        batch_target_wav = batch_info.labels[...,self.ref_mic].to(self.device)  # (B,L)
        batch_wav_len_list = batch_info.frame_mask_list

        # stft
        b_size, wav_len, channel_num = batch_mix_wav.shape
        batch_mix_wav = batch_mix_wav.transpose(-2, -1).contiguous().view(b_size*channel_num, wav_len)
        win_size, win_shift = int(self.sr*self.win_size), int(self.sr*self.win_shift)
        batch_mix_stft = torch.stft(
            batch_mix_wav,
            n_fft=self.fft_num,
            hop_length=win_shift,
            win_length=win_size,
            window=torch.hann_window(win_size).to(self.device),
            return_complex=False)   # (BM,F,T,2)
        batch_target_stft = torch.stft(
            batch_target_wav,
            n_fft=self.fft_num,
            hop_length=win_shift,
            win_length=win_size,
            window=torch.hann_window(win_size).to(self.device),
            return_complex=False)  # (B,F,T,2)
        batch_frame_list = []

        for i in range(len(batch_wav_len_list)):
            curr_frame_num = (batch_wav_len_list[i]-win_size+win_size)//win_shift+1  # center case
            batch_frame_list.append(curr_frame_num)

        _, freq_num, seq_len, _ = batch_mix_stft.shape
        batch_mix_stft = batch_mix_stft.view(b_size, -1, freq_num, seq_len, 2)

        if self.is_compress:  # here only apply to target and bf as feat-compression has been applied within the network
            # target
            batch_target_mag, batch_target_phase = torch.norm(batch_target_stft, dim=-1)**0.5, torch.atan2(
                batch_target_stft[..., -1], batch_target_stft[..., 0])
            batch_target_stft = torch.stack((batch_target_mag * torch.cos(batch_target_phase),
                                             batch_target_mag * torch.sin(batch_target_phase)), dim=-1)

        # convert to formats: (B,T,F,M,2) for mix, (B,T,F,2) for target and bf
        batch_mix_stft = batch_mix_stft.permute(0, 3, 2, 1, 4)
        batch_target_stft = batch_target_stft.transpose(1, 2)
        # net predict
        _, batch_spec_est = self.net(batch_mix_stft)  # (B,T,F,2), (B,T,F,2)

        # cal mse loss
        batch_mse_loss = self.spectral_loss(batch_spec_est, batch_target_stft, batch_frame_list)
        # cal metric loss

        if self.is_compress:
            batch_spec_mag, batch_spec_phase = torch.norm(batch_spec_est, dim=-1)**2.0,\
                                               torch.atan2(batch_spec_est[...,-1], batch_spec_est[...,0])
            batch_spec_est = torch.stack((batch_spec_mag*torch.cos(batch_spec_phase),
                                          batch_spec_mag*torch.sin(batch_spec_phase)), dim=-1)
            batch_target_mag, batch_target_phase = torch.norm(batch_target_stft, dim=-1) ** 2.0, \
                                               torch.atan2(batch_target_stft[...,-1], batch_target_stft[...,0])
            batch_target_stft = torch.stack((batch_target_mag * torch.cos(batch_target_phase),
                                          batch_target_mag * torch.sin(batch_target_phase)), dim=-1)
        batch_ref_mix_stft = _complex_from_ri(batch_mix_stft[...,self.ref_mic,:].transpose(1,2))  # (B,F,T,2)
        batch_spec_est = _complex_from_ri(batch_spec_est.transpose(1,2))  # (B,F,T,2)
        batch_target_stft = _complex_from_ri(batch_target_stft.transpose(1,2))  # (B,F,T,2)
        batch_mix_wav = torch.istft(batch_ref_mix_stft,
                                    n_fft=self.fft_num,
                                    hop_length=win_shift,
                                    win_length=win_size,
                                    window=torch.hann_window(win_size).to(self.device),
                                    length=wav_len)  # (B,L)
        batch_est_wav = torch.istft(batch_spec_est,
                                    n_fft=self.fft_num,
                                    hop_length=win_shift,
                                    win_length=win_size,
                                    window=torch.hann_window(win_size).to(self.device),
                                    length=wav_len)  # (B,L)
        batch_target_wav = torch.istft(batch_target_stft,
                                       n_fft=self.fft_num,
                                       hop_length=win_shift,
                                       win_length=win_size,
                                       window=torch.hann_window(win_size).to(self.device),
                                       length=wav_len)  # (B,L)

        loss_dict = {}
        loss_dict["mse_loss"] = batch_mse_loss.item()
        # create mask
        mask_list = []
        for id in range(b_size):
            mask_list.append(torch.ones((batch_wav_len_list[id])))
        wav_mask = torch.nn.utils.rnn.pad_sequence(mask_list, batch_first=True).to(batch_mix_stft.device)  # (B,L)
        batch_mix_wav, batch_target_wav, batch_est_wav = (batch_mix_wav * wav_mask).cpu().numpy(), \
                                                         (batch_target_wav * wav_mask).cpu().numpy(), \
                                                         (batch_est_wav * wav_mask).cpu().numpy()
        if "SISNR" in self.metric_options:
            unpro_score_list, pro_score_list = [], []
            for id in range(batch_mix_wav.shape[0]):
                unpro_score_list.append(cal_sisnr(id, batch_mix_wav, batch_target_wav, self.sr))
                pro_score_list.append(cal_sisnr(id, batch_est_wav, batch_target_wav, self.sr))
            unpro_score_list, pro_score_list = np.asarray(unpro_score_list), np.asarray(pro_score_list)
            unpro_sisnr_mean_score, pro_sisnr_mean_score = np.mean(unpro_score_list), np.mean(pro_score_list)
            loss_dict["unpro_metric"] = unpro_sisnr_mean_score
            loss_dict["pro_metric"] = pro_sisnr_mean_score
        if "NB-PESQ" in self.metric_options:
            unpro_score_list, pro_score_list = [], []
            for id in range(batch_mix_wav.shape[0]):
                unpro_score_list.append(cal_pesq(id, batch_mix_wav, batch_target_wav, self.sr))
                pro_score_list.append(cal_pesq(id, batch_est_wav, batch_target_wav, self.sr))
            unpro_score_list, pro_score_list = np.asarray(unpro_score_list), \
                                               np.asarray(pro_score_list)
            unpro_pesq_mean_score, pro_pesq_mean_score = np.mean(unpro_score_list), np.mean(pro_score_list)
            loss_dict["unpro_metric"] = unpro_pesq_mean_score
            loss_dict["pro_metric"] = pro_pesq_mean_score
        if "ESTOI" in self.metric_options:
            unpro_score_list, pro_score_list = [], []
            for id in range(batch_mix_wav.shape[0]):
                unpro_score_list.append(cal_stoi(id, batch_mix_wav, batch_target_wav, self.sr))
                pro_score_list.append(cal_stoi(id, batch_est_wav, batch_target_wav, self.sr))
            unpro_score_list, pro_score_list = np.asarray(unpro_score_list), \
                                               np.asarray(pro_score_list)
            unpro_estoi_mean_score, pro_estoi_mean_score = np.mean(unpro_score_list), np.mean(pro_score_list)
            loss_dict["unpro_metric"] = unpro_estoi_mean_score
            loss_dict["pro_metric"] = pro_estoi_mean_score
        return loss_dict


    def _train_batch(self, batch_info):
        batch_mix_wav = batch_info.feats.to(self.device)  # (B,L,M)
        batch_bf_wav = batch_info.bfs.to(self.device)  # (B,L)
        batch_target_wav = batch_info.labels[..., self.ref_mic].to(self.device)  # (B,L), only ref-mic is selected
        batch_wav_len_list = batch_info.frame_mask_list

        # stft
        b_size, wav_len, channel_num = batch_mix_wav.shape
        batch_mix_wav = batch_mix_wav.transpose(-2, -1).contiguous().view(b_size * channel_num, wav_len)
        win_size, win_shift = int(self.sr * self.win_size), int(self.sr * self.win_shift)
        batch_mix_stft = torch.stft(
            batch_mix_wav,
            n_fft=self.fft_num,
            hop_length=win_shift,
            win_length=win_size,
            window=torch.hann_window(win_size).to(batch_mix_wav.device),
            return_complex=False)  # (BM,F,T,2)
        batch_bf_stft = torch.stft(
            batch_bf_wav,
            n_fft=self.fft_num,
            hop_length=win_shift,
            win_length=win_size,
            window=torch.hann_window(win_size).to(batch_mix_wav.device),
            return_complex=False)  # (B,F,T,2)
        batch_target_stft = torch.stft(
            batch_target_wav,
            n_fft=self.fft_num,
            hop_length=win_shift,
            win_length=win_size,
            window=torch.hann_window(win_size).to(batch_mix_wav.device),
            return_complex=False)  # (B,F,T,2)
        batch_frame_list = []
        for i in range(len(batch_wav_len_list)):
            curr_frame_num = (batch_wav_len_list[i] - win_size + win_size) // win_shift + 1
            batch_frame_list.append(curr_frame_num)

        _, freq_num, seq_len, _ = batch_mix_stft.shape
        batch_mix_stft = batch_mix_stft.view(b_size, -1, freq_num, seq_len, 2)
        if self.is_compress:  # here only apply to target and bf as feat-compression has been applied within the network
            # target
            batch_target_mag, batch_target_phase = torch.norm(batch_target_stft, dim=-1)**0.5, torch.atan2(
                batch_target_stft[..., -1], batch_target_stft[..., 0])
            batch_target_stft = torch.stack((batch_target_mag * torch.cos(batch_target_phase),
                                             batch_target_mag * torch.sin(batch_target_phase)), dim=-1)
            # bf
            batch_bf_mag, batch_bf_phase = torch.norm(batch_bf_stft, dim=-1)**0.5, torch.atan2(
                batch_bf_stft[..., -1], batch_bf_stft[..., 0])
            batch_bf_stft = torch.stack((batch_bf_mag * torch.cos(batch_bf_phase),
                                         batch_bf_mag * torch.sin(batch_bf_phase)), dim=-1)

        # convert to formats: (B,T,F,M,2) for mix, (B,T,F,2) for target and bf
        batch_mix_stft = batch_mix_stft.permute(0, 3, 2, 1, 4)
        batch_bf_stft = batch_bf_stft.transpose(1, 2)
        batch_target_stft = batch_target_stft.transpose(1, 2)

        # ----------------------debug --------------------
        assert batch_mix_stft.shape[-1] == 2
        assert batch_mix_stft.ndim == 5

        with torch.enable_grad():
            batch_bf_est, batch_spec_est = self.net(batch_mix_stft)  # (B,T,F,2), (B,T,F,2)

        # ----------------------debug --------------------
        assert batch_bf_est.shape[-1] == 2
        assert batch_spec_est.shape[-1] == 2
        # beamforming loss
        resi = batch_bf_est - batch_bf_stft      # (B,T,F,2)
        batch_spatial_loss = self.spatial_loss(resi, batch_frame_list)
        # batch_spatial_loss = self.spectral_loss(batch_bf_est, batch_bf_stft, batch_frame_list)
        # reconstruction loss
        batch_spectral_loss = self.spectral_loss(batch_spec_est, batch_target_stft, batch_frame_list)
        batch_loss = self.spatial_weight*batch_spatial_loss + self.spectral_weight*batch_spectral_loss
        # params update
        self.update_params(batch_loss)
        loss_dict = {}
        loss_dict["spatial_bf_loss"] = batch_spatial_loss.item()
        loss_dict["spectral_loss"] = batch_spectral_loss.item()
        return loss_dict


    def _run_one_epoch(self, epoch, val_opt=False):
        # training phase
        if not val_opt:
            data_loader = self.train_dataloader
            total_bf_loss, total_sp_loss = 0., 0.
            start_time = time.time()
            for batch_id, batch_info in enumerate(data_loader.get_data_loader()):
                loss_dict = self._train_batch(batch_info)
                total_bf_loss += loss_dict["spatial_bf_loss"]
                total_sp_loss += loss_dict["spectral_loss"]
                if batch_id % self.print_freq == 0:
                    logger_print(
                        "Epoch:{:d}, Iter:{:d}, Average bf loss:{:.4f}, Average spectral loss:{:.4f}, Time: {:d}ms/batch".
                            format(epoch+1, int(batch_id), total_bf_loss/(batch_id+1), total_sp_loss/(batch_id+1),
                                                                    int(1000*(time.time()-start_time)/(batch_id+1))))
            return total_sp_loss / (batch_id+1)
        else:  # validation phase
            data_loder = self.val_dataloader
            total_sp_loss, total_pro_metric_loss, total_unpro_metric_loss = 0., 0., 0.
            start_time = time.time()
            for batch_id, batch_info in enumerate(data_loder.get_data_loader()):
                loss_dict = self._val_batch(batch_info)
                assert len(self.metric_options) == 1, "only one metric is supported to output in the val phase"
                total_sp_loss += loss_dict["mse_loss"]
                total_unpro_metric_loss += loss_dict["unpro_metric"]
                total_pro_metric_loss += loss_dict["pro_metric"]
                if batch_id % self.print_freq == 0:
                    logger_print(
                        "Epoch:{:d}, Iter:{:d}, Average spectral loss:{:.4f}, Average unpro metric score:{:.4f}, "
                        "Average pro metric score:{:.4f}, Time: {:d}ms/batch".
                            format(epoch+1, int(batch_id), total_sp_loss/(batch_id+1), total_unpro_metric_loss/(batch_id+1),
                                   total_pro_metric_loss/(batch_id+1), int(1000*(time.time()-start_time)/(batch_id+1))))
            return total_sp_loss / (batch_id+1), total_pro_metric_loss / (batch_id + 1)

    def update_params(self, loss):
        self.optimizer.zero_grad()
        loss.backward()
        if self.gradient_norm >= 0.0:
            nn.utils.clip_grad_norm_(self.net.parameters(), self.gradient_norm)
        has_nan_inf = 0
        for params in self.net.parameters():
            if params.requires_grad:
                has_nan_inf += torch.sum(torch.isnan(params.grad))
                has_nan_inf += torch.sum(torch.isinf(params.grad))
        if has_nan_inf == 0:
            self.optimizer.step()

## Main

In [ ]:
import os
import torch
import warnings
warnings.filterwarnings("ignore")
torch.set_warn_always(False)


root_train = "/content/drive/MyDrive/YoavAndItayShared/Speech/DataSet_train"
root_val   = "/content/drive/MyDrive/YoavAndItayShared/Speech/DataSet_valid"

loss_save_path  = "/content/drive/MyDrive/YoavAndItayShared/Speech/train_model_folder/loss"
model_best_path = "/content/drive/MyDrive/YoavAndItayShared/Speech/train_model_folder/BestModels"
ckpt_path       = "/content/drive/MyDrive/YoavAndItayShared/Speech/train_model_folder/Checkpoint"

os.makedirs(loss_save_path, exist_ok=True)
os.makedirs(model_best_path, exist_ok=True)
os.makedirs(ckpt_path, exist_ok=True)

train_set = InstanceDataset(
    mix_file_path=os.path.join(root_train, "mixture"),
    bf_file_path=os.path.join(root_train, "mvdr_out"),
    target_file_path=os.path.join(root_train, "clean_mc"),
    mix_json_path=os.path.join(root_train, "mix.json"),
    bf_json_path=os.path.join(root_train, "bf.json"),
    target_json_path=os.path.join(root_train, "target.json"),
    batch_size=8,
    is_check=True,
    is_shuffle=True,
    is_variance_norm=True,
    is_chunk=True,
    chunk_length=4.0,
    sr=16000,
)

valid_set = InstanceDataset(
    mix_file_path=os.path.join(root_val, "mixture"),
    bf_file_path=os.path.join(root_val, "mvdr_out"),
    target_file_path=os.path.join(root_val, "clean_mc"),
    mix_json_path=os.path.join(root_val, "mix.json"),
    bf_json_path=os.path.join(root_val, "bf.json"),
    target_json_path=os.path.join(root_val, "target.json"),
    batch_size=8,
    is_check=True,
    is_shuffle=True,
    is_variance_norm=True,
    is_chunk=True,
    chunk_length=4.0,
    sr=16000,
)

data = {
    "train_loader": InstanceDataloader(train_set, shuffle=False, num_workers=6, pin_memory=True, drop_last=False),
    "val_loader":   InstanceDataloader(valid_set, shuffle=False, num_workers=6, pin_memory=True, drop_last=False),
}

net = TaylorBeamformer(
    k1=[1, 3],
    k2=[2, 3],
    ref_mic=0,
    c=64,
    embed_dim=64,
    fft_num=320,
    order_num=3,
    kd1=5,
    cd1=64,
    d_feat=256,
    dilations=[1, 2, 5, 9],
    group_num=2,
    hid_node=64,
    M=5,
    rnn_type="LSTM",
    intra_connect="cat",
    inter_connect="cat",
    out_type="mapping",
    bf_type="embedding",
    norm2d_type="BN",
    norm1d_type="BN",
    is_compress=False,
    is_total_separate=False,
    is_u2=True,
    is_1dgate=True,
    is_squeezed=False,
    is_causal=True,
    is_param_share=False
)

optimizer = torch.optim.Adam(net.parameters(), lr=5e-4, betas=(0.9, 0.999), weight_decay=1e-7)

args = {
    "optimizer": {
        "lr": 5e-4,
        "gradient_norm": 5.0,
        "epochs": 60,
        "halve_lr": True,
        "early_stop": False,
        "halve_freq": 2,
        "early_stop_freq":6,
        "print_freq": 20,
        "metric_options": ["ESTOI"],
    },
    "loss_function": {
        "path": "utils.loss",
        "spectral": {"classname": "ComMagEuclideanLoss"},
        "spatial_weight": 1.0,
        "spectral_weight": 1.0,
        "alpha": 0.5,
        "l_type": "L2",
    },
    "signal": {
        "sr": 16000,
        "win_size": 0.02,
        "win_shift": 0.01,
        "fft_num": 320,
        "is_compress": False,
        "ref_mic": 0,
    },
    "path": {
        "is_checkpoint": True,
        "is_resume_reload": True,
        "checkpoint_load_path": ckpt_path,
        "checkpoint_load_filename": "/content/drive/MyDrive/YoavAndItayShared/Speech/train_model_folder/Checkpoint/Epoch_53_TaylorBeamformer_taylorbeamformer_ckpt.pth.tar",  # fill only if resume
        "loss_save_path": loss_save_path,
        "model_best_path": model_best_path,
    }
}


save_name_dict = {
    "loss_filename": "taylorbeamformer_loss.mat",
    "best_model_filename": "taylorbeamformer_best.pth",
    "checkpoint_filename": "taylorbeamformer_ckpt.pth.tar",
}


solver = Solver(data=data, net=net, optimizer=optimizer, save_name_dict=save_name_dict, args=args)
solver.train()

checking.................
checking finished..............
checking.................
checking finished..............
checking.................
checking finished..............
checking.................
checking finished..............
Begin to train....
------------------------------------------------------------------------------------------
Epoch id:54, Training phase, Start time:2026-03-01 14:03:29
Epoch:54, Iter:0, Average bf loss:155.8975, Average spectral loss:4.8825, Time: 50505ms/batch
Epoch:54, Iter:20, Average bf loss:64.2442, Average spectral loss:3.2295, Time: 5583ms/batch
Epoch:54, Iter:40, Average bf loss:52.0883, Average spectral loss:3.1104, Time: 4385ms/batch
Epoch:54, Iter:60, Average bf loss:47.3953, Average spectral loss:3.1014, Time: 3880ms/batch
Epoch:54, Iter:80, Average bf loss:45.5654, Average spectral loss:3.1508, Time: 3789ms/batch
Epoch:54, Iter:100, Average bf loss:44.5374, Average spectral loss:3.1152, Time: 3617ms/batch
Epoch:54, Iter:120, Average bf loss:43

KeyboardInterrupt: 

## Clean Cache

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import gc
import torch

# delete model and clear cache
try:
    del net
except:
    pass
torch.cuda.empty_cache()
gc.collect()